# Pancreatic Islet Pericyte Study — Computational Analysis Notebook
## PancDB Human Adult Control Dataset: QC, Integration, and ECM/BM Analysis

---

## Key Variables Reference

### QC and Filtering
| Variable | Type | Description |

| `hadata.obs["total_counts"]` | float | Total UMI counts per cell |
| `hadata.obs["n_genes_by_counts"]` | int | Number of genes detected per cell |
| `hadata.obs["log_counts"]` | float | Log10(total_counts + 1) |
| `hadata.obs["log_genes"]` | float | Log10(n_genes_by_counts + 1) |
| `hadata.obs["qc_score"]` | float | Composite QC score (sum of log_counts_z + log_genes_z) |
| `hadata.obs["doublet"]` | int | DoubletDetection call: 1 = doublet, 0 = singlet |
| `hadata.obs["doublet_score"]` | float | Continuous doublet probability score |

### Cell Type Annotations
| Variable | Type | Description |
|---|---|---|
| `hadata.obs["cell_type"]` | categorical | Original cell type labels from PancDB |
| `hadata.obs["cell_type_grouped"]` | categorical | Cell type with endocrine subtypes collapsed into "endocrine cell" |
| `hadata.obs["donor_id"]` | categorical | Donor sample identifier |
| `hadata.obs["disease_state"]` | categorical | Control vs. disease condition |
| `hadata.obs["age"]` | float/int | Donor age |

### Embeddings
| Key | Object | Description |
|---|---|---|
| `obsm["X_umap"]` | hadata / ts_adata | scVI-derived UMAP |
| `obsm["X_scvi_umap"]` | ts_adata | Tabula Sapiens scVI UMAP |
| `obsm["Concord"]` | hadata_clean_removed | CONCORD latent embedding (full dataset) |
| `obsm["Concord_UMAP"]` | hadata_clean_removed / mesenchymal_dr | UMAP computed on Concord embedding |
| `obsm["Concord_UMAPm"]` | mesenchymal | UMAP recomputed on mesenchymal subset only |
| `obsm["Concord_mergedmesenchyme"]` | merged_mesenchymepulbic | CONCORD embedding of merged 4-dataset object |
| `obsm["Concord_mergedmesenchyme_UMAP"]` | merged_mesenchymepulbic | UMAP of merged mesenchymal CONCORD embedding |

### Clustering
| Variable | Type | Description |
|---|---|---|
| `mesenchymal_dr.obs["mesenchymal_leiden"]` | categorical | Leiden sub-clusters within mesenchymal cells; final labels are MSL1 and MSL2 |
| `mesenchymal_endopancdb.obs["mesenchymal_leiden"]` | categorical | Leiden clusters on stromal subset (pre-MSL merging) |

### Gene Sets (ECM/BM)
| Variable | Type | Description |
|---|---|---|
| `ecm_genes` | list | All genes from NABA_CORE_MATRISOME.v2025.1.Hs.gmt |
| `ecm_filtered` | list | ECM genes with BM genes removed (non-overlapping) |
| `bmall_genes` | list | All genes from NABA_BASEMENT_MEMBRANES.v2025.1.Hs.gmt |
| `bm_genes` | list | First gene set entry from BM GMT file |
| `corematrisome_genes` | list | First gene set entry from core matrisome GMT file |
| `ven_diagram` | list | Curated 28-gene panel of canonical mesenchymal marker genes |
| `scored_gene_sets` | list | Gene set names successfully scored and stored in .obs |

### Gene Set Scores (stored in .obs)
| Column | Object | Description |
|---|---|---|
| `"ECM_Matrisome"` | hadata_clean_removed / adata | sc.tl.score_genes() score for ECM_Matrisome |
| `"Basement_Membrane"` | hadata_clean_removed / adata | sc.tl.score_genes() score for BM gene set |
| `"BM_score"` / `"ECM_score"` | hadata_clean_removed | Raw mean expression score (pre-z-score) |
| `"BM_zscore"` / `"ECM_zscore"` | hadata_clean_removed | Z-score normalized version of above |
| `"Pericyte_signature"` | hca/ts/shp mesenchyme objects | MSL2 (cluster1sig_n) gene list score |
| `"Fibro_signature"` | hca/ts/shp mesenchyme objects | MSL1 (cluster0sig_n) gene list score |

### DEG Results
| Variable | Type | Description |
|---|---|---|
| `cluster0sig_n` | list | Top 50 marker genes for MSL1 (Fibroblast-like cluster) |
| `cluster1sig_n` | list | Top 50 marker genes for MSL2 (Pericyte-like cluster) |
| `ductal_bm` | DataFrame | Wilcoxon DEG results for duct epithelial cells restricted to BM genes |
| `top_bm_genes` | list | Top 20 BM-enriched genes in duct epithelial cells |
| `deg_df` | DataFrame | Full DEG table for all mesenchymal Leiden clusters (exported to CSV) |

### Layers
| Layer | Description |
|---|---|
| `"counts"` | Raw counts stored before normalization (hadata) |
| `"log1p"` | Log1p-normalized expression matrix (hca_mesenchyme, ts_mesenchyme, merged object) |

---

## Software and Key Package Versions

| Package | Version | Use |
|---|---|---|
| `scanpy` | ≥ 1.9 | Core scRNA-seq analysis |
| `anndata` | ≥ 0.9 | Data structure |
| `concord` | 2025/2026 (Gartner Lab, UCSF) | Batch integration and UMAP |
| `doubletdetection` | v4.2 | Doublet calling |
| `torch` | ≥ 2.0 | Backend for CONCORD |
| `scipy` | ≥ 1.10 | Z-score normalization, hierarchical clustering |
| `seaborn` | ≥ 0.12 | Boxplots and KDE plots |
| `matplotlib` | ≥ 3.7 | All figure export (PDF/SVG for Illustrator) |
| `pandas` | ≥ 2.0 | Data wrangling and Prism table export |
| `numpy` | ≥ 1.24 | Numerical operations |

---

## Output Directories

| Path | Contents |
|---|---|
| `/Users/kmeneses/Desktop/Fig_plots/` | Main figure panels (PDF + SVG) |
| `/Users/kmeneses/Desktop/Figure_1_plots/SI/` | Supplementary figure panels |
| `/Users/kmeneses/Desktop/SI_plots/` | Additional SI plots (PNG) |
| `/Users/kmeneses/Desktop/SI/` | Scanpy violin plot outputs |

In [ ]:
import concord as ccd
import scanpy as sc
import anndata as ad
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import matplotlib as mpl
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Dataset: PancDB Human Adult Control

In [ ]:
#uploading human adult subset dataset from PancDB website 
hadata = sc.read_h5ad('/Volumes/Samsung_4TB/Public_human_RNAdasets/PANCDB_adult_control.h5ad')
hadata

# Dataset: PancDB Human Adult Control — Quality Control Assessment
We loaded the PancDB human adult control dataset and performed a multi-metric quality control assessment across donors. First, we calculated standard QC metrics (total counts and genes detected per cell) and visualized their distributions per donor using violin plots. To enable cross-donor comparison, we log-transformed and z-score normalized both metrics, then combined them into a composite QC score per cell.

In [ ]:
# Calculate per-cell QC metrics: total counts, genes detected, etc.
sc.pp.calculate_qc_metrics(
    hadata,
    inplace=True
)

In [ ]:
# Visualize count and gene distributions per donor to spot outlier samples
sc.pl.violin(
    hadata,
    ["n_genes_by_counts", "total_counts"],
    groupby="donor_id",
    jitter=0.4,
    multi_panel=True,
    rotation=45
)

In [ ]:
import numpy as np

# log transform (stabilizes scale)
hadata.obs["log_counts"] = np.log10(hadata.obs["total_counts"] + 1)
hadata.obs["log_genes"] = np.log10(hadata.obs["n_genes_by_counts"] + 1)


# z-score normalize each metric
qc_metrics = ["log_counts", "log_genes"]

for m in qc_metrics:
    hadata.obs[f"{m}_z"] = (
        (hadata.obs[m] - hadata.obs[m].mean()) / hadata.obs[m].std()
    )

# Sum z-scores into a single composite QC score per cell
# Higher score = higher quality cell (more counts + more genes detected)
hadata.obs["qc_score"] = (hadata.obs["qc_score"] = (
    hadata.obs["log_counts_z"] +
    hadata.obs["log_genes_z"] 
)

In [ ]:
# Summarize sample-level quality as median QC score per donor
qc_summary = hadata.obs.groupby("donor_id")["qc_score"].median().sort_values()

import matplotlib.pyplot as plt

plt.figure(figsize=(5,2.5))

qc_summary.plot(kind="bar")

plt.ylabel("Median QC score")
plt.xlabel("Sample")
plt.title("Median cell quality per donor")

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# KDE plot to compare the full QC score distribution across donors
# Overlapping curves reveal donors with skewed or bimodal quality profiles
plt.figure(figsize=(5,3))

for donor in hadata.obs["donor_id"].unique():
    subset = hadata.obs[hadata.obs["donor_id"] == donor]
    sns.kdeplot(subset["qc_score"], label=donor, alpha=0.3)

plt.xlabel("QC score")
plt.title("QC distribution per donor")
plt.legend(bbox_to_anchor=(1.05,1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Log-log scatter of total counts vs genes detected
# Well-behaved cells follow a linear trend; outliers may indicate doublets or empty droplets

plt.figure(figsize=(4,3))

sns.scatterplot(
    data=hadata.obs,
    x="total_counts",
    y="n_genes_by_counts",
    s=3,
    alpha=0.3
)

plt.xscale("log")
plt.yscale("log")

plt.xlabel("Total counts")
plt.ylabel("Number of genes")
plt.title("Library complexity relationship")

plt.tight_layout()
plt.show()

In [ ]:
# Fraction of cells in which each gene is detected (count > 0)
gene_detection = (hadata.X > 0).mean(axis=0)

In [ ]:
# Set global rcParams for publication-quality figures (Arial font, PDF/SVG compatible)


# =========================
# STYLE (Illustrator-friendly)
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# COUNT CELLS PER SAMPLE
# =========================
sample_counts = (
    hadata.obs["donor_id"]
    .value_counts()
    .sort_index()   # keeps donor order clean
)

# Convert to dataframe (optional, useful for export)
df_counts = sample_counts.reset_index()
df_counts.columns = ["donor_id", "n_cells"]

# =========================
# Bar chart of cell counts per donor
# =========================
plt.figure(figsize=(4,2.5))

plt.bar(
    df_counts["donor_id"],
    df_counts["n_cells"]
)

plt.xticks(rotation=45, ha="right")
plt.ylabel("Number of cells")
plt.xlabel("Sample")
plt.title("Cells per sample")

plt.tight_layout()

plt.show()

In [ ]:
df_counts

In [ ]:
# ── 1. UMAP — ORIGINAL DATA (pre-filtering) ──────────────────────────────────
# Plot UMAP colored by metadata variables to inspect dataset structure before QC filtering
# Rasterize scatter points to reduce PDF/SVG file size for Illustrator
# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})
plt.rcParams["figure.figsize"] = (2, 2)
# =========================
# PLOT (let scanpy build grid)
# =========================
fig = sc.pl.embedding(
    hadata,
    basis="X_umap",
    color=['age', 'donor_id', 'cell_type', 'disease_state'],
    ncols=2,
    size=1,
    wspace=0.5,                 # 🔥 smaller = lighter file
    frameon=True,
    legend_loc='right',
    show=False,
    return_fig=True           # 🔥 IMPORTANT
)

# =========================
# 🔥 RASTERIZE POINTS (KEY FIX)
# =========================
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

# =========================
# CLEAN AXES + BORDER
# =========================
for ax in fig.axes:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    
    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_1_plots/SI/pandc_hadault_OG"

fig.savefig(f"{out_base}.pdf", dpi=600, bbox_inches="tight")  # 🔥 BEST for Illustrator
fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:


# =========================
# COUNT CELLS PER SAMPLE
# =========================
sample_counts = (
    hadata.obs["donor_id"]
    .value_counts()
    .sort_index()
)

df_counts = sample_counts.reset_index()
df_counts.columns = ["donor_id", "n_cells"]

# =========================
# PLOT
# =========================
plt.figure(figsize=(4,2.5))

bars = plt.bar(
    df_counts["donor_id"],
    df_counts["n_cells"]
)

# 🔥 ADD VALUE LABELS ABOVE BARS
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height,
        f'{int(height)}',
        ha='center',
        va='bottom',
        fontsize=6,
        rotation=90   # optional: rotate if crowded
    )

plt.xticks(rotation=45, ha="right")
plt.ylabel("Number of cells")
plt.xlabel("Sample")
plt.title("Cells per sample")

plt.tight_layout()
plt.show()

In [ ]:
# Remove low-quality cells: too few counts (likely empty droplets)
# Remove high-count outliers (likely doublets or damaged cells)
sc.pp.filter_cells(hadata, min_counts=100)
sc.pp.filter_cells(hadata, max_counts=20000)

In [ ]:
hadata

In [ ]:
# Replot UMAP after basic count filtering to confirm filter impact on structure

# =========================
# =========================
mpl.rcParams.update({
    "svg.fonttype": "none",   # editable text in Illustrator
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8
})

# =========================
# UMAP plot
# =========================
fig = sc.pl.umap(
    hadata,
    color=['age', 'donor_id', 'cell_type', 'disease_state'],
    ncols=1,
    frameon=False,
    size=8,          # better for print
    wspace=0.3,
    vmin='p1',        # robust scaling (for continuous vars like age)
    vmax='p99',
    return_fig=True
)

# =========================
# Save (vector + high resolution)
# =========================
save_path = "/Users/kmeneses/Desktop/Fig_plots/UMAP_metadata_SI_pancdb_afterminQC"

fig.savefig(f"{save_path}.svg", dpi=600, bbox_inches="tight")
fig.savefig(f"{save_path}.pdf", dpi=600, bbox_inches="tight")



In [ ]:
hadata_clean_removed

In [ ]:
# Recompute neighborhood graph and UMAP on hadata_clean_removed
# (cells removed in a prior filtering step not shown here)
sc.pp.neighbors(hadata_clean_removed)
sc.tl.umap(hadata_clean_removed)

In [ ]:
# ── 7. UMAP — PUBLICATION STYLE (no bold, clean) ─────────────────────────────
# Final UMAP export with non-bold fonts for publication formatting (scVI)
sc.pl.embedding(
    hadata_clean_removed,
    basis="X_umap",
    color=['age', 'donor_id', 'cell_type', 'disease_state'],
    ncols=2,
    size=1,
    wspace=0.5,                 # 🔥 smaller = lighter file
    frameon=True,
    legend_loc='right',
    show=False,
    return_fig=True           # 🔥 IMPORTANT
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import scanpy as sc

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "font.weight": "normal", 
    "axes.titleweight": "normal",   # 🔥 no bold
    "axes.labelweight": "normal", 
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})
plt.rcParams["figure.figsize"] = (2, 2)
# =========================
# PLOT (let scanpy build grid)
# =========================
fig = sc.pl.embedding(
    hadata,
    basis="X_umap",
    color=['age', 'donor_id', 'cell_type', 'disease_state'],
    ncols=2,
    size=1,
    wspace=0.5,                 # 🔥 smaller = lighter file
    frameon=True,
    legend_loc='right',
    show=False,
    return_fig=True           # 🔥 IMPORTANT
)

# =========================
# 🔥 RASTERIZE POINTS (KEY FIX)
# =========================
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

# =========================
# CLEAN AXES + BORDER
# =========================
for ax in fig.axes:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.title.set_fontweight("normal")
    ax.xaxis.label.set_fontweight("normal")
    ax.yaxis.label.set_fontweight("normal")
    
    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_1_plots/SI/pandc_hadault_miniQC"

fig.savefig(f"{out_base}.pdf", dpi=600, bbox_inches="tight")  # 🔥 BEST for Illustrator
fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
# ── 8. PER-DONOR QC SUMMARY TABLE ────────────────────────────────────────────
# Aggregate median counts, genes, and composite QC score per donor
# Sorted by QC score to identify lowest-quality samples

qc = (
    hadata.obs
    .groupby("donor_id")
    .agg(
        median_counts=("total_counts", "median"),
        median_genes=("n_genes_by_counts", "median"),
        median_qc_score=("qc_score", "median")
    )
)

qc.sort_values("median_qc_score")

In [ ]:
# Store raw counts in a separate layer before normalization (for downstream use)
hadata.layers['counts'] = hadata.X.copy()

In [ ]:
sc.pp.normalize_total(hadata, target_sum=1e4)
sc.pp.log1p(hadata)

In [ ]:

# Save per-donor violin plots of gene counts and total counts for SI figures

# Set save directory
sc.settings.figdir = "/Users/kmeneses/Desktop/SI"

# Make SVG editable in Illustrator
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["pdf.fonttype"] = 42

# Optional: improve fonts
mpl.rcParams["font.family"] = "Arial"
sns.set_style("white")  # no grid background
mpl.rcParams["axes.grid"] = False
# Plot
sc.pl.violin(
    hadata,
    keys="n_genes_by_counts",
    groupby="donor_id",
    rotation=90,
    stripplot=False,
    show=True,
    save="vp_pancdb_qcN_genes.svg"
)

In [ ]:
hadata

In [ ]:
import scanpy as sc
import matplotlib as mpl

# Set save directory
sc.settings.figdir = "/Users/kmeneses/Desktop/SI"

# Make SVG editable in Illustrator
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["pdf.fonttype"] = 42

# Optional: improve fonts
mpl.rcParams["font.family"] = "Arial"
sns.set_style("white")  # no grid background
mpl.rcParams["axes.grid"] = False
# Plot
sc.pl.violin(
    hadata,
    keys="total_counts",
    groupby="donor_id",
    rotation=90,
    stripplot=False,
    show=True,
    save="vp_pancdb_qcN_cgenes.svg"
)

In [ ]:
import doubletdetection
%matplotlib inline

# ── 11. DOUBLET DETECTION ─────────────────────────────────────────────────────
# Doublet detection using DoubletDetection v4.2 (Gayoso & Shor, 2020)
# https://github.com/JonathanShor/DoubletDetection
#
# DoubletDetection uses a boosting classifier approach:
#   - Artificially creates synthetic doublets by combining real cell pairs
#   - Clusters all cells (real + synthetic) using Leiden algorithm
#   - Classifies cells as doublets if they co-cluster with synthetic doublets
#     across n_iters=10 independent boosting rounds
#
# Key parameters:
#   n_iters=10            — number of boosting rounds (more = more robust)
#   clustering_algorithm  — Leiden community detection for cluster assignment
#   standard_scaling=True — z-score normalize before clustering
#   pseudocount=0.1       — added to counts before log transform to avoid log(0)
#   p_thresh=1e-16        — p-value threshold for doublet classification
#   voter_thresh=0.5      — fraction of iterations that must call a cell a doublet
#   n_jobs=-1             — parallelized across all available CPU cores

In [ ]:
clf = doubletdetection.BoostClassifier(
    n_iters=10, 
    clustering_algorithm="leiden", 
    standard_scaling=True,
    pseudocount=0.1,
    n_jobs=-1,
)
doublets = clf.fit(hadata.X).predict(p_thresh=1e-16, voter_thresh=0.5)
doublet_score = clf.doublet_score()

In [ ]:
# Store doublet calls and scores in obs for downstream filtering and visualization

hadata.obs["doublet"] = doublets
hadata.obs["doublet_score"] = doublet_score

In [ ]:
# ── 12. DOUBLET SCORE DISTRIBUTION — FULL RANGE ───────────────────────────────
# Histogram of doublet scores with summary statistics annotated
# Helps determine whether the score threshold is appropriate

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# DATA
# =========================
scores = hadata.obs["doublet_score"].dropna()

# =========================
# STATS
# =========================
mean_val = scores.mean()
median_val = scores.median()
std_val = scores.std()

print(f"Mean: {mean_val:.3f}")
print(f"Median: {median_val:.3f}")
print(f"Std: {std_val:.3f}")

# =========================
# PLOT
# =========================
plt.figure(figsize=(2, 1.5))

plt.hist(
    scores,
    bins=150,
    range=(0, 10),   # zoom to meaningful range
    edgecolor="black"
)

# -------------------------
# annotate stats
# -------------------------
textstr = (
    f"Mean = {mean_val:.2f}\n"
    f"Median = {median_val:.2f}\n"
    f"SD = {std_val:.2f}"
)

plt.text(
    0.98, 0.95,
    textstr,
    transform=plt.gca().transAxes,
    ha='right',
    va='top'
)

plt.xlabel("Doublet score")
plt.ylabel("Cell count")
plt.title("Doublet score distribution")

plt.tight_layout()

# =========================
# EXPORT
# =========================
out = "/Users/kmeneses/Desktop/Figure_1_plots/SI/doublet_score_hist_stats"

plt.savefig(f"{out}.pdf", bbox_inches="tight")
plt.savefig(f"{out}.svg", bbox_inches="tight")

plt.show()

In [ ]:
# ── 13. DOUBLET SCORE DISTRIBUTION — ZOOMED (99th percentile) ────────────────
# Clip to 99th percentile to zoom into the bulk of the distribution
# Reveals fine structure obscured by high-score outliers
# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# DATA
# =========================
scores = hadata.obs["doublet_score"].dropna()

# =========================
# PLOT (ZOOMED)
# =========================
vmax = np.percentile(scores, 99)

plt.figure(figsize=(2, 1.5))

plt.hist(
    scores[scores <= vmax],
    bins=500,
    edgecolor="black"
)

# ✅ apply all formatting FIRST
plt.xlim(0, vmax)
plt.ylim(0, 2000)

plt.xlabel("Doublet score")
plt.ylabel("Cell count")
plt.title("Doublet score distribution")

plt.tight_layout()

# ✅ THEN save
out = "/Users/kmeneses/Desktop/Figure_1_plots/SI/doublet_score_hist"

plt.savefig(
    f"{out}.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"{out}.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ── 14. REMOVE UNKNOWN CELL TYPES ────────────────────────────────────────────
# Exclude cells labeled "unknown" in cell_type — unresolved identity
# # Define clusters to remove
clusters_to_remove = ["unknown"]  # Example: cluster labels as strings or ints

# Subset the data to exclude those clusters
hadata = hadata[~hadata.obs["cell_type"].isin(clusters_to_remove)].copy()

In [ ]:
# ── 15. DOUBLET SCORE PANEL — ZOOMED + FULL LOG SCALE ────────────────────────
# Side-by-side comparison: zoomed linear view and full log-scale view
# Confirms doublet score threshold is appropriate across the full range

scores = hadata.obs["doublet_score"].dropna()

fig, axes = plt.subplots(1, 2, figsize=(4, 1.5))

# -------------------------
# LEFT: zoomed
# -------------------------
axes[0].hist(
    scores,
    bins=150,
    range=(0, 10),
    edgecolor="black"
)
axes[0].set_xlim(0, 10)
axes[0].set_title("Zoom (0–10)")

# -------------------------
# RIGHT: full (log)
# -------------------------
axes[1].hist(
    scores,
    bins=200,
    edgecolor="black"
)
axes[1].set_yscale("log")
axes[1].set_title("Full (log scale)")

# -------------------------
# labels
# -------------------------
for ax in axes:
    ax.set_xlabel("Doublet score")
    ax.set_ylabel("Cell count")

plt.tight_layout()

# 🔥 SAVE (must be AFTER layout)
out = "/Users/kmeneses/Desktop/Figure_1_plots/SI/doublet_score_panel"

fig.savefig(f"{out}.svg", bbox_inches="tight")
fig.savefig(f"{out}.pdf", bbox_inches="tight")

plt.show()

In [ ]:
# ── 16. UMAP — DOUBLET CALLS + SCORES ────────────────────────────────────────
# Visualize doublet calls and scores on UMAP to confirm spatial distribution
# Doublets should not cluster in biologically meaningful populations
# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "font.weight": "normal", 
    "axes.titleweight": "normal",   # 🔥 no bold
    "axes.labelweight": "normal", 
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})
plt.rcParams["figure.figsize"] = (1, 1)
# =========================
# PLOT (let scanpy build grid)
# =========================
fig = sc.pl.embedding(
    hadata,
    basis="X_umap",
    color=['doublet', 'doublet_score'],
    ncols=2,
    cmap='cool',
    size=1,
    frameon=True,
    legend_loc='right',
    vmax=[None, 100],   # 👈 only affects doublet_score
    show=False,
    return_fig=True
)

# =========================
# 🔥 RASTERIZE POINTS (KEY FIX)
# =========================
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

# =========================
# CLEAN AXES + BORDER
# =========================
for ax in fig.axes:
    ax.title.set_fontweight("normal")
    ax.xaxis.label.set_fontweight("normal")
    ax.yaxis.label.set_fontweight("normal")
    
    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_1_plots/SI/pandc_hadault_miniQC_doublet"

fig.savefig(f"{out_base}.pdf", dpi=600, bbox_inches="tight")  # 🔥 BEST for Illustrator
fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
sc.pl.embedding(hadata, basis='X_umap', ncols=2, cmap='viridis', color=["doublet", "doublet_score", "cell_type", "donor_id"])


In [ ]:
# Summary statistics of doublet scores across all cells

hadata.obs["doublet_score"].describe()

In [ ]:
# ── 18. FILTER DOUBLETS ───────────────────────────────────────────────────────
# Remove predicted doublets (doublet == 1), retain only singlets
hadata_clean = hadata[hadata.obs["doublet"] != 1].copy()

In [ ]:
# ── 19. CELLS PER SAMPLE AFTER DOUBLET REMOVAL ───────────────────────────────
# Recount cells per donor after doublet filtering to assess impact on sample sizes
# =========================
# STYLE (Illustrator-friendly)
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# COUNT CELLS PER SAMPLE
# =========================
sample_counts = (
    hadata_clean.obs["donor_id"]
    .value_counts()
    .sort_index()   # keeps donor order clean
)

# Convert to dataframe (optional, useful for export)
df_counts = sample_counts.reset_index()
df_counts.columns = ["donor_id", "n_cells"]

# =========================
# PLOT
# =========================
plt.figure(figsize=(4,2.5))

plt.bar(
    df_counts["donor_id"],
    df_counts["n_cells"]
)

plt.xticks(rotation=45, ha="right")
plt.ylabel("Number of cells")
plt.xlabel("Sample")
plt.title("Cells per sample")

plt.tight_layout()

# =========================
# SAVE
# =========================


plt.show()

In [ ]:
# Summary table of final cell counts per donor
df_counts

In [ ]:
hadata_clean

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import scanpy as sc

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

scores = hadata.obs["doublet_score"].dropna()

# =========================
# 🔥 AUTO THRESHOLD (robust)
# =========================
threshold = np.percentile(scores, 95)   # adjust if needed

# assign doublets
hadata.obs["doublet"] = hadata.obs["doublet_score"] > threshold

print(f"Threshold: {threshold:.2f}")
print(hadata.obs["doublet"].value_counts(normalize=True))

# =========================
# 🔥 FIGURE: HIST + LOG + UMAP
# =========================
fig = plt.figure(figsize=(6, 2))

# -------------------------
# 1. Histogram (zoom)
# -------------------------
ax1 = plt.subplot(1, 3, 1)

ax1.hist(scores, bins=150, range=(0, 10), edgecolor="black")
ax1.axvline(threshold, linestyle="--")
ax1.set_xlim(0, 10)

ax1.set_title("Zoom (0–10)")
ax1.set_xlabel("Score")
ax1.set_ylabel("Count")

# -------------------------
# 2. Full distribution (log)
# -------------------------
ax2 = plt.subplot(1, 3, 2)

ax2.hist(scores, bins=200)
ax2.set_yscale("log")
ax2.axvline(threshold, linestyle="--")

ax2.set_title("Full (log)")
ax2.set_xlabel("Score")

# -------------------------
# 3. UMAP QC
# -------------------------
ax3 = plt.subplot(1, 3, 3)

sc.pl.embedding(
    hadata,
    basis="X_umap",
    color="doublet_score",
    cmap="cool",
    size=1,
    frameon=False,
    show=False,
    ax=ax3
)

ax3.set_title("UMAP (score)")

# rasterize points (small file)
for coll in ax3.collections:
    coll.set_rasterized(True)

# =========================
# SAVE
# =========================
plt.tight_layout()

out = "/Users/kmeneses/Desktop/Figure_1_plots/SI/doublet_QC_panel"

plt.savefig(f"{out}.pdf", bbox_inches="tight")
plt.savefig(f"{out}.svg", bbox_inches="tight")

plt.show()

In [ ]:
for t in [3, 5, 8, 10, 20, 40]:
    frac = (scores > t).mean()
    print(f"{t}: {frac:.3f}")

In [ ]:
hadata_clean

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import scanpy as sc

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "font.weight": "normal", 
    "axes.titleweight": "normal",   # 🔥 no bold
    "axes.labelweight": "normal", 
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})
plt.rcParams["figure.figsize"] = (1, 1)
# =========================
# PLOT (let scanpy build grid)
# =========================
fig = sc.pl.embedding(
    hadata_clean,
    basis="X_umap",
    color=['doublet', 'doublet_score'],
    ncols=2,
    cmap='cool',
    size=1,  
    wspace=None,               # 🔥 smaller = lighter file
    frameon=True,
    legend_loc='right',
    show=False,
    return_fig=True           # 🔥 IMPORTANT
)

# =========================
# 🔥 RASTERIZE POINTS (KEY FIX)
# =========================
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

# =========================
# CLEAN AXES + BORDER
# =========================
for ax in fig.axes:
    ax.title.set_fontweight("normal")
    ax.xaxis.label.set_fontweight("normal")
    ax.yaxis.label.set_fontweight("normal")
    
    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_1_plots/SI/pandc_hadault_miniQC_doubletremoval_just1notscore"

fig.savefig(f"{out_base}.pdf", dpi=600, bbox_inches="tight")  # 🔥 BEST for Illustrator
fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")

plt.show()

## 12. Batch Integration with CONCORD

To correct for donor-specific batch effects while preserving biological variation, 
we applied CONCORD (Contrastive Unified Representation for Single-Cell Omics Data;
Hu et al., 2025; Gartner Lab, UCSF; https://github.com/Gartner-Lab/Concord).

CONCORD is a deep learning-based integration method that uses contrastive learning 
to align cells across batches by learning a low-dimensional embedding that separates 
technical variation (donor/batch) from biological signal (cell type, disease state). 
Unlike graph-based methods (e.g., Harmony, Scanorama), CONCORD operates directly on 
normalized expression values and does not require a PCA initialization.

Key advantages for this dataset:
- Designed for datasets with many donors and unbalanced sample sizes
- Preserves fine-grained cell type structure during integration
- Outputs a corrected low-dimensional embedding used for 
  downstream UMAP visualization, clustering, and differential analysis

Input:  hadata_clean — log1p-normalized counts, doublets removed, 
        unknown cell types excluded
Output: CONCORD embedding stored in hadata_clean.obsm["Concord"]

In [ ]:
# Set device to cpu or to gpu (if your torch has been set up correctly to use GPU), for mac you can use either torch.device('mps') or torch.device('cpu')
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# (Optional) Select top variably expressed/accessible features for analysis (other methods besides seurat_v3 available)
feature_list = ccd.ul.select_features(hadata_clean, n_top_features=2000, flavor='seurat_v3')


# If integrating data across batch, simply add the domain_key argument to indicate the batch key in adata.obs
cur_ccd = ccd.Concord(adata=hadata_clean, input_feature=feature_list, use_faiss=False, domain_key='donor_id', device=device) 

# Encode data, saving the latent embedding in adata.obsm['Concord']
cur_ccd.fit_transform(output_key='Concord')

In [ ]:
ccd.ul.run_umap(hadata_clean, source_key='Concord', result_key='Concord_UMAP', n_components=2, n_neighbors=15, min_dist=0.1, metric='euclidean')

# Plot the UMAP embeddings
color_by = ['donor_id', 'cell_type'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    hadata_clean, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='upper right')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import scanpy as sc

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "font.weight": "normal", 
    "axes.titleweight": "normal",   # 🔥 no bold
    "axes.labelweight": "normal", 
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})
plt.rcParams["figure.figsize"] = (1, 1)
# =========================
# PLOT (let scanpy build grid)
# =========================
fig = sc.pl.embedding(
    hadata_clean,
    basis="Concord_UMAP",
    color=['donor_id', 'cell_type'],
    ncols=2,
    cmap='viridis',
    size=1,  
    wspace=None,               # 🔥 smaller = lighter file
    frameon=True,
    legend_loc='right',
    show=False,
    return_fig=True           # 🔥 IMPORTANT
)

# =========================
# 🔥 RASTERIZE POINTS (KEY FIX)
# =========================
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

# =========================
# CLEAN AXES + BORDER
# =========================
for ax in fig.axes:
    ax.title.set_fontweight("normal")
    ax.xaxis.label.set_fontweight("normal")
    ax.yaxis.label.set_fontweight("normal")
    
    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_1_plots/SI/pandc_hadault_concord_qc1_doulbetpredonlty"

fig.savefig(f"{out_base}.pdf", dpi=600, bbox_inches="tight")  # 🔥 BEST for Illustrator
fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
hadata_clean

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import scanpy as sc

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "font.weight": "normal", 
    "axes.titleweight": "normal",   # 🔥 no bold
    "axes.labelweight": "normal", 
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})
plt.rcParams["figure.figsize"] = (2, 2)
# =========================
# PLOT (let scanpy build grid)
# =========================
fig = sc.pl.embedding(
    hadata_clean,
    basis="Concord_UMAP",
    color=['age', 'disease_state'],
    ncols=1,
    size=1,
    wspace=0.1,                 # 🔥 smaller = lighter file
    frameon=True,
    legend_loc='right',
    show=False,
    return_fig=True           # 🔥 IMPORTANT
)

# =========================
# 🔥 RASTERIZE POINTS (KEY FIX)
# =========================
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

# =========================
# CLEAN AXES + BORDER
# =========================
for ax in fig.axes:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.title.set_fontweight("normal")
    ax.xaxis.label.set_fontweight("normal")
    ax.yaxis.label.set_fontweight("normal")
    
    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_1_plots/SI/pandc_hadault_miniQC_concordintegration_agrecontol"

fig.savefig(f"{out_base}.pdf", dpi=600, bbox_inches="tight")  # 🔥 BEST for Illustrator
fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")

plt.show()

 # ── 13. DONOR-LEVEL QC OUTLIER DETECTION ──────────────────────────────────────
 Identifies low-quality donor samples based on aggregate QC metrics.

# Approach:
  For each donor, we compute median values of three QC metrics:
    - total_counts     : sequencing depth per cell
    - n_genes_by_counts: transcriptome complexity per cell
    - pct_counts_mt    : mitochondrial read fraction (proxy for cell stress/damage)

  Each metric is then z-score normalized across donors to enable
  cross-sample comparison on a common scale.

# Outlier rule (z-score thresholds):
 A donor is flagged as low-quality if ANY of the following are true:
  - median_counts_z  < -1.5  (unusually low sequencing depth)
  - median_genes_z   < -1.5  (unusually low gene complexity)
  - median_mt_z      >  1.5  (unusually high mitochondrial content)

# Output:
  - low_quality_samples : list of donor IDs flagged for potential exclusion
  - qc_summary_per_donor.csv : full per-donor QC table sorted by median counts,
    saved to SI figures directory for reporting

In [ ]:
import pandas as pd
from scipy.stats import zscore

# =========================
# 1. Build QC table per donor
# =========================
qc_full = (
    hadata_clean.obs
    .groupby("donor_id")
    .agg(
        median_counts=("total_counts", "median"),
        median_genes=("n_genes_by_counts", "median"),
        median_mt=("pct_counts_mt", "median")
    )
)

# =========================
# 2. Compute Z-scores
# =========================
qc_full["median_counts_z"] = zscore(qc_full["median_counts"])
qc_full["median_genes_z"]  = zscore(qc_full["median_genes"])
qc_full["median_mt_z"]     = zscore(qc_full["median_mt"])

# =========================
# 3. Define outlier rule
# =========================
outlier_mask = (
    (qc_full["median_counts_z"] < -1.5) |
    (qc_full["median_genes_z"] < -1.5) |
    (qc_full["median_mt_z"] > 1.5)
)

qc_full["low_quality"] = outlier_mask

# =========================
# 4. Get flagged samples
# =========================
low_quality_samples = qc_full[qc_full["low_quality"]].index.tolist()

print("Samples flagged as low-quality:")
print(low_quality_samples)

# =========================
# 5. Optional: sort table
# =========================
qc_full_sorted = qc_full.sort_values("median_counts")
out = "/Users/kmeneses/Desktop/Figure_1_plots/SI/qc_summary_per_donor.csv"

qc_full_sorted.to_csv(out)
print(qc_full_sorted)

# Filtering of low quality donors

In [ ]:
# Define clusters to remove
clusters_to_remove = ["HPAP093"]  # Example: cluster labels as strings or ints

# Subset the data to exclude those clusters
hadata_clean_removed = hadata_clean[~hadata_clean.obs["donor_id"].isin(clusters_to_remove)].copy()

In [ ]:
hadata_clean_removed

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import scanpy as sc

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "font.weight": "normal", 
    "axes.titleweight": "normal",   # 🔥 no bold
    "axes.labelweight": "normal", 
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})
plt.rcParams["figure.figsize"] = (2, 2)
# =========================
# PLOT (let scanpy build grid)
# =========================
fig = sc.pl.embedding(
    hadata_clean_removed,
    basis="Concord_UMAP",
    color=['donor_id', 'cell_type'],
    ncols=1,
    size=1,
    wspace=0.1,                 # 🔥 smaller = lighter file
    frameon=True,
    legend_loc='right',
    show=False,
    return_fig=True           # 🔥 IMPORTANT
)

# =========================
# 🔥 RASTERIZE POINTS (KEY FIX)
# =========================
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

# =========================
# CLEAN AXES + BORDER
# =========================
for ax in fig.axes:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.title.set_fontweight("normal")
    ax.xaxis.label.set_fontweight("normal")
    ax.yaxis.label.set_fontweight("normal")
    
    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_1_plots/SI/pandc_hadault_miniQC_CI_no93_dob"

fig.savefig(f"{out_base}.pdf", dpi=600, bbox_inches="tight")  # 🔥 BEST for Illustrator
fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
# Define clusters to remove
clusters_to_remove = ["HPAP067"]  # Example: cluster labels as strings or ints

# Subset the data to exclude those clusters
hadata_clean_removed = hadata_clean_removed[~hadata_clean_removed.obs["donor_id"].isin(clusters_to_remove)].copy()

In [ ]:
hadata_clean_removed

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import scanpy as sc

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "font.weight": "normal", 
    "axes.titleweight": "normal",   # 🔥 no bold
    "axes.labelweight": "normal", 
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})
plt.rcParams["figure.figsize"] = (2, 2)
# =========================
# PLOT (let scanpy build grid)
# =========================
fig = sc.pl.embedding(
    hadata_clean_removed,
    basis="Concord_UMAP",
    color=['donor_id', 'cell_type'],
    ncols=1,
    size=1,
    wspace=0.1,                 # 🔥 smaller = lighter file
    frameon=True,
    legend_loc='right',
    show=False,
    return_fig=True           # 🔥 IMPORTANT
)

# =========================
# 🔥 RASTERIZE POINTS (KEY FIX)
# =========================
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

# =========================
# CLEAN AXES + BORDER
# =========================
for ax in fig.axes:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.title.set_fontweight("normal")
    ax.xaxis.label.set_fontweight("normal")
    ax.yaxis.label.set_fontweight("normal")
    
    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_1_plots/SI/pandc_hadault_miniQC_CI_no9367_dv"

fig.savefig(f"{out_base}.pdf", dpi=600, bbox_inches="tight")  # 🔥 BEST for Illustrator
fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
# Define clusters to remove
clusters_to_remove = [ "HPAP027",]  # Example: cluster labels as strings or ints

# Subset the data to exclude those clusters
hadata_clean_removed = hadata_clean_removed[~hadata_clean_removed.obs["donor_id"].isin(clusters_to_remove)].copy()

In [ ]:
hadata_clean_removed

## 14. UMAP Recomputation and Metadata Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import scanpy as sc

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "font.weight": "normal", 
    "axes.titleweight": "normal",   # 🔥 no bold
    "axes.labelweight": "normal", 
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})
plt.rcParams["figure.figsize"] = (2, 2)
# =========================
# PLOT (let scanpy build grid)
# =========================
fig = sc.pl.embedding(
    hadata_clean_removed,
    basis="Concord_UMAP",
    color=['donor_id', 'cell_type'],
    ncols=1,
    size=1,
    wspace=0.1,                 # 🔥 smaller = lighter file
    frameon=True,
    legend_loc='right',
    show=False,
    return_fig=True           # 🔥 IMPORTANT
)

# =========================
# 🔥 RASTERIZE POINTS (KEY FIX)
# =========================
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

# =========================
# CLEAN AXES + BORDER
# =========================
for ax in fig.axes:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.title.set_fontweight("normal")
    ax.xaxis.label.set_fontweight("normal")
    ax.yaxis.label.set_fontweight("normal")
    
    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_1_plots/SI/pandc_hadault_miniQC_CI_no936727_db"

fig.savefig(f"{out_base}.pdf", dpi=600, bbox_inches="tight")  # 🔥 BEST for Illustrator
fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
# ── 1. UMAP — ORIGINAL DATA (pre-filtering) ──────────────────────────────────
# Plot UMAP colored by metadata variables to inspect dataset structure before QC filtering
# Rasterize scatter points to reduce PDF/SVG file size for Illustrator
# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})
plt.rcParams["figure.figsize"] = (2, 2)
# =========================
# PLOT (let scanpy build grid)
# =========================
fig = sc.pl.embedding(
    hadata,
    basis="X_umap",
    color=['age', 'donor_id', 'cell_type', 'disease_state'],
    ncols=2,
    size=1,
    wspace=0.5,                 # 🔥 smaller = lighter file
    frameon=True,
    legend_loc='right',
    show=False,
    return_fig=True           # 🔥 IMPORTANT
)

# =========================
# 🔥 RASTERIZE POINTS (KEY FIX)
# =========================
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

# =========================
# CLEAN AXES + BORDER
# =========================
for ax in fig.axes:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    
    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_1_plots/SI/pandc_hadault_OG"

fig.savefig(f"{out_base}.pdf", dpi=600, bbox_inches="tight")  # 🔥 BEST for Illustrator
fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")

plt.show()

## 15. ECM and Basement Membrane Gene Set Scoring

To assess extracellular matrix (ECM) and basement membrane (BM) program activity 
across pancreatic cell types, we used curated gene sets from the NABA Matrisome 
database (MSigDB v2025.1, Hs):

- **NABA_CORE_MATRISOME** — broad ECM gene set covering all matrisome components
- **NABA_BASEMENT_MEMBRANES** — genes specifically associated with basement membrane assembly

Gene sets were loaded from GMT files, deduplicated, and BM genes were subtracted 
from the core matrisome list to create a non-overlapping ECM_Matrisome gene set.
Per-cell gene set scores were computed using Scanpy's `sc.tl.score_genes()`, which
calculates the average expression of the gene set relative to a random control gene set.

Results were visualized as dot plots grouped by cell type, with endocrine subtypes 
collapsed into a single "endocrine cell" category for clarity. Dot size reflects the 
fraction of cells expressing the gene set; color reflects relative enrichment score.

We then performed Wilcoxon rank-sum differential expression within BM genes to identify 
the top BM genes enriched in duct epithelial cells relative to all other cell types, 
and visualized the top hits as a dot plot across ductal, endothelial, and mesenchymal cells.

In [ ]:
# ── 1. GMT LOADER FUNCTIONS ───────────────────────────────────────────────────
# Utility functions for parsing GMT-format gene set files (MSigDB standard format)
# GMT format: set_name \t description \t gene1 \t gene2 \t ...
# --- core loaders ---
def load_gmt(path):
    """Return dict: set_name -> list of genes."""
    gs = {}
    with open(path, "r") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 3:
                set_name = parts[0]
                genes = [g for g in parts[2:] if g]
                gs[set_name] = genes
    return gs


def genes_from_gmt(path, set_name):
    """Return gene list for one named set."""
    with open(path, "r") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 3 and parts[0] == set_name:
                return [g for g in parts[2:] if g]
    return []


def all_genes_flat(path):
    """Return one flat unique list of all genes in the GMT file."""
    seen, out = set(), []
    with open(path, "r") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            for g in parts[2:]:
                if g and g not in seen:
                    seen.add(g)
                    out.append(g)
    return out


# --- use them ---
gmt_path = "/Users/kmeneses/Downloads/NABA_CORE_MATRISOME.v2025.1.Hs.gmt"

gene_sets = load_gmt(gmt_path)               # dict of sets
corematrisome_genes = genes_from_gmt(gmt_path, list(gene_sets.keys())[0])  # pick first set name
ecm_genes = all_genes_flat(gmt_path)         # one big list, de-duplicated

print(f"Found {len(gene_sets)} sets; first set has {len(corematrisome_genes)} genes; total unique genes = {len(ecm_genes)}")

In [ ]:
# --- core loaders ---
def load_gmt(path):
    """Return dict: set_name -> list of genes."""
    gs = {}
    with open(path, "r") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 3:
                set_name = parts[0]
                genes = [g for g in parts[2:] if g]
                gs[set_name] = genes
    return gs


def genes_from_gmt(path, set_name):
    """Return gene list for one named set."""
    with open(path, "r") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 3 and parts[0] == set_name:
                return [g for g in parts[2:] if g]
    return []


def all_genes_flat(path):
    """Return one flat unique list of all genes in the GMT file."""
    seen, out = set(), []
    with open(path, "r") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            for g in parts[2:]:
                if g and g not in seen:
                    seen.add(g)
                    out.append(g)
    return out


# --- use them ---
gmt_path = "/Users/kmeneses/Downloads/NABA_BASEMENT_MEMBRANES.v2025.1.Hs.gmt"

gene_sets = load_gmt(gmt_path)               # dict of sets
bm_genes = genes_from_gmt(gmt_path, list(gene_sets.keys())[0])  # pick first set name
bmall_genes = all_genes_flat(gmt_path)         # one big list, de-duplicated

print(f"Found {len(gene_sets)} sets; first set has {len(bm_genes)} genes; total unique genes = {len(bmall_genes)}")

In [ ]:
# ── 4. SUBTRACT BM GENES FROM ECM SET ────────────────────────────────────────
# Remove basement membrane genes from the core matrisome list to create
# a non-overlapping ECM_Matrisome gene set for independent scoring
# Convert to sets for fast operations
bm_set = set(bmall_genes)
ecm_set = set(ecm_genes)

# Remove BM genes from ECM
ecm_filtered = list(ecm_set - bm_set)

print(f"Original ECM genes: {len(ecm_genes)}")
print(f"Filtered ECM genes: {len(ecm_filtered)}")
print(f"Removed overlap: {len(ecm_set & bm_set)}")

In [ ]:
# ── 5. SCORE GENE SETS — ALL CELL TYPES ──────────────────────────────────────
# Compute per-cell enrichment scores for ECM_Matrisome and Basement_Membrane
# using sc.tl.score_genes() — scores average expression of the gene set
# relative to a randomly sampled control gene set of equal size

mpl.rcParams["svg.fonttype"] = "none"   # keeps text editable
mpl.rcParams["font.family"] = "Arial"   # optional but recommended
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

# Define properly as dictionary
gene_sets = {
    "ECM_Matrisome": ecm_filtered,
    "Basement_Membrane": bmall_genes
}

# Score gene sets
for gs_name, gene_list in gene_sets.items():

    valid_genes = [g for g in gene_list if g in hadata_clean_removed.var_names]

    if len(valid_genes) == 0:
        print(f"⚠️ Skipping {gs_name} (no valid genes found)")
        continue

    sc.tl.score_genes(
        hadata_clean_removed,
        gene_list=valid_genes,
        score_name=gs_name,
        use_raw=False
    )

# Collect successfully scored sets
scored_gene_sets = [
    gs for gs in gene_sets.keys()
    if gs in hadata_clean_removed.obs.columns
]

print("Scored gene sets:", scored_gene_sets)

# Plot
dp = sc.pl.dotplot(
    hadata_clean_removed,
    var_names=scored_gene_sets,
    groupby="cell_type",
    title="Relative Enrichment of ECM Programs",
    colorbar_title="Z-score",
    cmap="viridis",
    standard_scale="var",
    return_fig=True   # <-- important
)

# Save as SVG
dp.savefig("/Users/kmeneses/Desktop/Fig_plots/ECM_gene_sethadata_dotplot_clean.svg", dpi= 600, bbox_inches="tight")



In [ ]:
import matplotlib as mpl
import scanpy as sc
import matplotlib.pyplot as plt
# Rescore on hadata_clean_removed (scores already computed above, reuse)
# Plot with biologically meaningful cell type order and magma colormap
order = [
# =========================
# Style (keeps text editable in Illustrator)
# =========================
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

# =========================
# 1. Collapse endocrine cells
# =========================
endocrine_cells = [
    "alpha cell",
    "beta cell",
    "delta cell",
    "PP cell",
    "epsilon cell"
]

hadata_clean_removed.obs["cell_type_grouped"] = hadata_clean_removed.obs["cell_type"].replace(
    {ct: "endocrine cell" for ct in endocrine_cells}
)

# =========================
# 2. Define gene sets
# =========================
gene_sets = {
    "ECM_Matrisome": ecm_filtered,
    "Basement_Membrane": bmall_genes
}

# =========================
# 3. Score gene sets
# =========================
for gs_name, gene_list in gene_sets.items():

    valid_genes = [g for g in gene_list if g in hadata_clean_removed.var_names]

    if len(valid_genes) == 0:
        print(f"⚠️ Skipping {gs_name} (no valid genes found)")
        continue

    sc.tl.score_genes(
        hadata_clean_removed,
        gene_list=valid_genes,
        score_name=gs_name,
        use_raw=False
    )

# =========================
# 4. Keep valid gene sets
# =========================
scored_gene_sets = [
    gs for gs in gene_sets.keys()
    if gs in hadata_clean_removed.obs.columns
]

print("Scored gene sets:", scored_gene_sets)

# =========================
# 5. Define biological order
# =========================
order = [
    "mesenchymal cell",
    "endothelial cell",
    "duct epithelial cell",
    "acinar cell",
    "beta cell",
    "alpha cell", 
    "delta cell", 
    "PP cell",
    "epsilon cell",
    "leukocyte"
    
]

# =========================
# 6. Plot dotplot
# =========================
dp = sc.pl.dotplot(
    hadata_clean_removed,
    var_names=scored_gene_sets,
    groupby="cell_type",
    categories_order=order,
    title="Relative Enrichment of ECM Programs",
    colorbar_title="Relative score",   # 🔥 fix label
    cmap="magma",                  # 🔥 better contrast
    standard_scale="var",
    figsize=(2.25, 3), 
    return_fig=True
)

# =========================
# 7. Save
# =========================
dp.savefig(
    "/Users/kmeneses/Desktop/Fig_plots/ECM_gene_set_dotplot_grouped.svg",
    dpi=600,
    bbox_inches="tight"
)

In [ ]:
# ── 7. DOTPLOT WITH GROUPED ENDOCRINE CATEGORY ───────────────────────────────
# Replot using cell_type_grouped (endocrine subtypes collapsed) on hadata_clean
# Cleaner view for main figure presentation

# =========================
# Style (keeps text editable in Illustrator)
# =========================
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42



# =========================
# 5. Define biological order
# =========================
order = [
    "mesenchymal cell",
    "endothelial cell",
    "duct epithelial cell",
    "endocrine cell",
    "acinar cell",
    "leukocyte",
    
    
]

# =========================
# 6. Plot dotplot
# =========================
dp = sc.pl.dotplot(
    hadata_clean,
    var_names=scored_gene_sets,
    groupby="cell_type_grouped",
    categories_order=order,
    title="Relative Enrichment of ECM Programs",
    colorbar_title="Relative score",   # 🔥 fix label
    cmap="magma",                  # 🔥 better contrast
    standard_scale="var",
    return_fig=True
)

# =========================
# 7. Save
# =========================
dp.savefig(
    "/Users/kmeneses/Desktop/Fig_plots/ECM_gene_set_dotplot_grouped_re.svg",
    dpi=600,
    bbox_inches="tight"
)

─ 8. DIFFERENTIAL EXPRESSION — DUCT EPITHELIAL CELLS (ALL GENES) ───────────
# Wilcoxon rank-sum test: duct epithelial cells vs. all other cell types
# Identifies genes broadly enriched in ductal cells across the full transcriptome

In [ ]:

sc.tl.rank_genes_groups(
    hadata_clean,
    groupby="cell_type",
    groups=["duct epithelial cell"],   # exact name
    reference="rest",
    method="wilcoxon",
    use_raw=False
)

In [ ]:
# Quick violin plot to inspect expression of key BM genes in ductal cells

sc.pl.violin(
    hadata_clean[hadata_clean.obs["cell_type"] == "duct epithelial cell"],
    keys=["COL18A1","LAMC2","LAMB3", "LAMA3", "AGRN", "LAMC1"],
    stripplot=True,
    use_raw=False
)

In [ ]:
# ── 9. DIFFERENTIAL EXPRESSION — DUCT EPITHELIAL CELLS (BM GENES ONLY) ───────
# Repeat Wilcoxon test but restrict the variable mask to BM genes only
# This focuses the ranking on basement membrane-specific transcriptional enrichment
bm_mask = hadata_clean_removed.var_names.isin(bmall_genes)

sc.tl.rank_genes_groups(
    hadata_clean_removed,
    groupby="cell_type",
    groups=["duct epithelial cell"],
    reference="rest",
    method="wilcoxon",
    use_raw=False,
    mask_var=bm_mask
)

ductal_bm = sc.get.rank_genes_groups_df(
    hadata_clean_removed,
    group="duct epithelial cell"
)

ductal_bm.head(50)



In [ ]:
# ── 10. SUBSET TO STROMAL + DUCTAL CELLS FOR VISUALIZATION ───────────────────
# Focus on the three cell types most relevant to BM production and remodeling:
# duct epithelial, endothelial, and mesenchymal cells

cell_subset = [
    "duct epithelial cell",
    "endothelial cell",
    "mesenchymal cell"
]

adata_sub = hadata_clean_removed[
    hadata_clean_removed.obs["cell_type"].isin(cell_subset)
].copy()

In [ ]:
# ── 11. DOTPLOT — TOP BM GENES IN DUCTAL vs STROMAL CELLS ────────────────────
# Visualize top 20 BM-enriched ductal genes across the three cell type subset
# Raw log1p expression shown (no z-scoring) so absolute expression levels are comparable


top_bm_genes = ductal_bm["names"].head(20).tolist()

# ---- Define fixed color limits (important for consistency) ----
vmin = 0
vmax = 3   # adjust depending on your dataset dynamic range

# Plot raw log1p heatmap
sc.pl.dotplot(
    adata_sub,
    var_names=top_bm_genes,
    groupby="cell_type",
    cmap="viridis",
    standard_scale=None,   # <-- no z-score
    swap_axes=False,
    figsize=(6,1.5),
    use_raw=False,         # assuming .X is log1p
    show=False,       # <-- remove legend
)

fig = plt.gcf()

fig.savefig(
    "/Users/kmeneses/Desktop/Fig_plots/BM_genhadatae_set_heatmap.svg",
    bbox_inches="tight",
    dpi=600
)


## 16. Mesenchymal and Endothelial Cell Subset Analysis

To characterize ECM-producing stromal populations, we subsetted mesenchymal and 
endothelial cells from the cleaned dataset and performed sub-clustering analysis. 
Using the CONCORD embedding for visualization, we applied Leiden clustering to 
identify transcriptionally distinct subpopulations within the stromal compartment.

Additionally, we computed mean ECM/BM gene set scores across all cell types and 
endocrine subtypes to contextualize stromal ECM enrichment relative to the broader 
pancreatic cell type landscape.

In [ ]:
# Isolate stromal populations (mesenchymal and endothelial) from the full
# cleaned dataset for focused sub-clustering and ECM analysis
CLUSTER_KEY   = "cell_type"
# Column containing clusters
CLUSTER_KEY = "cell_type"    # <-- change if needed

# Clusters you want to merge into a separate AnnData
keep_clusters = ["mesenchymal cell", "endothelial cell"]     # <-- EDIT: choose the two clusters you want

# Subset + merge them into a new AnnData
mesenchymal_endopancdb = hadata_clean_removed[hadata_clean_removed.obs[CLUSTER_KEY].astype(str).isin(keep_clusters)].copy()

print("Cells in merged subset:", mesenchymal_endopancdb.n_obs)

In [ ]:
# Plot cell type and doublet score on CONCORD UMAP to confirm subset integrity
color_by = ['cell_type', 'doublet_score'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    mesenchymal_endopancdb, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='none', 
    save_path='/Users/kmeneses/Desktop/SI_plots/Concord_UMAP_pancdbmesenchymalendothelialonly_ECMBMscore.png')

In [ ]:
color_by = ['cell_type', 'doublet_score', 'donor_id'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    mesenchymal_endopancdb, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='none')

In [ ]:
# ── 3. DOUBLET RATE PER CELL TYPE ────────────────────────────────────────────
# Cross-tabulate doublet calls by cell type (row-normalized)
# Confirms doublet removal was uniform across cell types

pd.crosstab(
    hadata.obs["cell_type"],
    hadata.obs["doublet"],
    normalize="index"
)

In [ ]:
# ── 4. LEIDEN SUB-CLUSTERING OF STROMAL SUBSET ───────────────────────────────
# Recompute neighbors and apply Leiden clustering within the stromal subset
# Resolution 0.22 chosen to yield interpretable coarse subpopulations
sc.pp.neighbors(mesenchymal_endopancdb)
sc.tl.leiden(mesenchymal_endopancdb, resolution=0.22, key_added='mesenchymal_leiden')

In [ ]:
# Visualize Leiden clusters and donor distribution on CONCORD UMAP
color_by = ['mesenchymal_leiden', 'donor_id'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    mesenchymal_endopancdb, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='upper right')

In [ ]:
# Check cell counts per Leiden cluster
mesenchymal_endopancdb.obs['mesenchymal_leiden'].value_counts()

In [ ]:
# MARKER GENES PER LEIDEN CLUSTER 
# Identify top marker genes per cluster using Wilcoxon rank-sum test
# Used to annotate or flag low-quality/ambiguous clusters
sc.tl.rank_genes_groups(mesenchymal_endopancdb, "mesenchymal_leiden", use_raw=False)
sc.pl.rank_genes_groups(mesenchymal_endopancdb, gene_symbols='features')

In [ ]:
# Cluster 5 flagged for removal based on marker gene inspection
# (low gene complexity, ambiguous identity, or likely contaminant)
remove_clusters = ["5"]
mesenchymal_endopancdb_reeee = mesenchymal_endopancdb[~mesenchymal_endopancdb.obs["mesenchymal_leiden"].isin(remove_clusters)].copy()

In [ ]:
color_by = ['mesenchymal_leiden', 'donor_id'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    mesenchymal_endopancdb_reeee, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='upper right')

In [ ]:
mesenchymal_endopancdb

In [ ]:
# Define your desired palette
celltype_colors = {
    "mesenchymal cell": "#FFD700",        # yellow
    "endothelial cell": "#808080"# gre             # optional
}

# Make sure categories match exactly
mesenchymal_endopancdb.obs["cell_type"] = mesenchymal_endopancdb.obs["cell_type"].astype("category")

# Assign colors in the order of categories
mesenchymal_endopancdb.uns["cell_type_colors"] = [
    celltype_colors[c] for c in mesenchymal_endopancdb.obs["cell_type"].cat.categories
]

# Plot
sc.pl.embedding(
    mesenchymal_endopancdb,
    basis="Concord_UMAP",   # or "umap"
    color="cell_type",
    size=10,
    frameon=False
)

In [ ]:
# SUBSET ENDOCRINE CELLS + COMPUTE MEAN ECM SCORES
# Isolate endocrine subtypes to compare ECM/BM gene set scores
# against stromal populations# 
endocrine_types = [
    "alpha cell",
    "beta cell",
    "delta cell",
    "PP cell", 
    "epsilon cell"
]

# -----------------------------
# SUBSET DATA
# -----------------------------
endo = hadata_clean_removed[
    hadata_clean_removed.obs["cell_type"].isin(endocrine_types)
].copy()

# -----------------------------
# CELL COUNTS
# -----------------------------
cell_counts = endo.obs["cell_type"].value_counts()

print("\nCells per endocrine type:")
print(cell_counts)

# -----------------------------
# MEAN ACROSS ALL ENDOCRINE CELLS
# -----------------------------
mean_scores_endo = endo.obs[scored_gene_sets].mean()

mean_scores_endo = mean_scores_endo.to_frame(name="Mean_score")

print("\nMean gene set scores (all endocrine cells pooled):")
print(mean_scores_endo)

In [ ]:
print(ecm_filtered)

In [ ]:
# -----------------------------
# BUILD MEAN MATRIX
# -----------------------------
mean_scores = (
    hadata_clean_removed.obs
    .groupby("cell_type")[scored_gene_sets]
    .mean()
)

print(mean_scores.head())

In [ ]:
color_by = ['mesenchymal_leiden', 'cell_type'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    mesenchymal_endopancdb, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='upper right')

## 17. Mesenchymal Cell Sub-clustering and Marker Analysis

Following isolation of mesenchymal cells from the stromal subset, we performed 
focused sub-clustering to identify transcriptionally distinct mesenchymal subpopulations. 


Leiden clustering (resolution=0.1) was applied on the CONCORD embedding, followed by 
iterative cluster inspection using marker gene analysis and QC metric overlays. 
Cluster 7 was removed as a low-quality population. Remaining clusters were merged into 
two biologically meaningful groups based on marker gene expression:
- **MSL1** — clusters 0, 1, 4, 5, 6 (broad mesenchymal/fibroblast identity)
- **MSL2** — clusters 2, 3 (distinct mesenchymal subpopulation)

Marker genes, ECM/BM gene set scores, and key mesenchymal identity genes were 
visualized on both CONCORD and scVI UMAP embeddings. DEGs between MSL1 and MSL2 
were computed using a Wilcoxon rank-sum test and exported for downstream analysis.
Mean ECM_Matrisome and Basement_Membrane scores were computed per cluster to 
quantify ECM program enrichment across mesenchymal subpopulations.

In [ ]:
# Column containing clusters
CLUSTER_KEY = "cell_type"    # <-- change if needed

# Clusters you want to merge into a separate AnnData
keep_clusters = ["mesenchymal cell"]     # <-- EDIT: choose the two clusters you want

# Subset + merge them into a new AnnData
mesenchymal = mesenchymal_endopancdb[mesenchymal_endopancdb.obs[CLUSTER_KEY].astype(str).isin(keep_clusters)].copy()

print("Cells in merged subset:", mesenchymal.n_obs)

In [ ]:
# Visualize ECM_Matrisome and Basement_Membrane scores on mesenchymal CONCORD UMAP
# vmax=0.5 caps colorscale to improve contrast; rasterized for Illustrator export
# 2. CREATE FULL-FRAME FIGURE WITH BORDER
# =========================
fig = plt.figure(figsize=(2.2, 2.2))

# We leave a tiny 1% margin so the black border line doesn't get clipped
ax = fig.add_axes([0.01, 0.01, 0.98, 0.98]) 

sc.pl.embedding(
    mesenchymal,
    basis="Concord_UMAP",
    color= ['ECM_Matrisome' , 'Basement_Membrane'],
    cmap='viridis',
    size=8,
    use_raw=False,
    frameon=True,  # 🔥 TURN THE FRAME BACK ON
    legend_loc=None,
    show=False,
    vmax=0.5,
    sort_order=True
)

# INSTEAD OF set_axis_off(), we just hide the text/notches:
ax.set_title("")
ax.set_xticks([])
ax.set_yticks([])


# Optional: Ensure the border line weight matches your Arial 8pt style
for spine in ax.spines.values():
    spine.set_linewidth(1)

# =========================
# 3. EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Fig_plots/UMAP_with_frame_mesencymal_BMscores_db"

# We still use pad_inches=0 to keep the file size tight to the box
plt.savefig(f"{out_base}.pdf", pad_inches=0, dpi=600)
plt.savefig(f"{out_base}.svg", format="svg", pad_inches=0)

plt.show()

In [ ]:
# Compute neighbors and apply Leiden clustering on full mesenchymal subset
# Initial pass before doublet score filtering
sc.pp.neighbors(mesenchymal)
sc.tl.leiden(mesenchymal, resolution=0.1, key_added='mesenchymal_leiden')

In [ ]:
color_by = ['mesenchymal_leiden', 'doublet_score'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    mesenchymal, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='upper right')

In [ ]:
# Filter out cells with doublet_score >= 20 to reduce technical noise
mesenchymal_dr = mesenchymal[mesenchymal.obs["doublet_score"] < 20].copy()

In [ ]:
sc.pp.neighbors(mesenchymal_dr)
sc.tl.leiden(mesenchymal_dr, resolution=0.1, key_added='mesenchymal_leiden')

In [ ]:
color_by = ['mesenchymal_leiden', 'doublet_score'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    mesenchymal_dr, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='upper right')

In [ ]:
mesenchymal_dr.obs['mesenchymal_leiden'].value_counts()

In [ ]:
# Wilcoxon rank-sum DEG test to identify top markers per Leiden cluster
# Used to annotate clusters and flag low-quality populations for removal

# =========================
# STYLE
# =========================
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["font.family"] = "Arial"

# =========================
# RUN DEG
# =========================
sc.tl.rank_genes_groups(
    mesenchymal_dr,
    "mesenchymal_leiden",
    use_raw=False
)

# =========================
# PLOT
# =========================
plt.figure(figsize=(3, 2))   # 🔥 force figure

sc.pl.rank_genes_groups(
    mesenchymal,
    gene_symbols="features",
    n_genes=20,
    show=False
)

# =========================
# GET FIG + SAVE
# =========================
fig = plt.gcf()   # 🔥 ALWAYS works

out_base = "/Users/kmeneses/Desktop/Fig_plots/mesenchymal_markersdr"

fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")
fig.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")



In [ ]:
sc.tl.rank_genes_groups(mesenchymal_dr, "mesenchymal_leiden", use_raw=False)
sc.pl.rank_genes_groups(mesenchymal_dr, gene_symbols='features')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import scanpy as sc

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "font.weight": "normal", 
    "axes.titleweight": "normal",   # 🔥 no bold
    "axes.labelweight": "normal", 
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})
plt.rcParams["figure.figsize"] = (3, 3)
# =========================
# PLOT (let scanpy build grid)
# =========================
fig = sc.pl.embedding(
    mesenchymal,
    basis="Concord_UMAP",
    color = ['doublet_score', 'cell_type', 'total_counts', 'n_genes_by_counts'], # Choose which variables you want to visualize
    ncols=3,
    cmap='cool',
    size=5,
    wspace=0.5,                 # 🔥 smaller = lighter file
    frameon=True,
    legend_loc='right',
    show=False,
    return_fig=True           # 🔥 IMPORTANT
)

# =========================
# 🔥 RASTERIZE POINTS (KEY FIX)
# =========================
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

# =========================
# CLEAN AXES + BORDER
# =========================
for ax in fig.axes:
 
    ax.title.set_fontweight("normal")
    ax.xaxis.label.set_fontweight("normal")
    ax.yaxis.label.set_fontweight("normal")
    
    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_1_plots/SI/pandc_mesenchymal_leidensluste"

fig.savefig(f"{out_base}.pdf", dpi=600, bbox_inches="tight")  # 🔥 BEST for Illustrator
fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import scanpy as sc

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "font.weight": "normal", 
    "axes.titleweight": "normal",   # 🔥 no bold
    "axes.labelweight": "normal", 
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})
plt.rcParams["figure.figsize"] = (3, 3)
# =========================
# PLOT (let scanpy build grid)
# =========================
fig = sc.pl.embedding(
    mesenchymal_dr,
    basis="Concord_UMAP",
    color = ['mesenchymal_leiden', 'doublet_score', 'cell_type', 'total_counts', 'n_genes_by_counts'], # Choose which variables you want to visualize
    ncols=3,
    size=5,
    wspace=0.5,                 # 🔥 smaller = lighter file
    frameon=True,
    legend_loc='right',
    show=False,
    return_fig=True           # 🔥 IMPORTANT
)

# =========================
# 🔥 RASTERIZE POINTS (KEY FIX)
# =========================
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

# =========================
# CLEAN AXES + BORDER
# =========================
for ax in fig.axes:
 
    ax.title.set_fontweight("normal")
    ax.xaxis.label.set_fontweight("normal")
    ax.yaxis.label.set_fontweight("normal")
    
    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_1_plots/SI/pandc_mesenchymal_leidenslustersdr"

fig.savefig(f"{out_base}.pdf", dpi=600, bbox_inches="tight")  # 🔥 BEST for Illustrator
fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
sc.pl.umap(mesenchymal_dr, color=["mesenchymal_leiden", "n_genes_by_counts"])

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import scanpy as sc

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "font.weight": "normal", 
    "axes.titleweight": "normal",   # 🔥 no bold
    "axes.labelweight": "normal", 
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})
plt.rcParams["figure.figsize"] = (3, 3)
# =========================
# PLOT (let scanpy build grid)
# =========================
fig = sc.pl.embedding(
    mesenchymal_dr,
    basis="X_umap",
    color = ['mesenchymal_leiden', 'doublet_score', 'cell_type', 'total_counts', 'n_genes_by_counts'], # Choose which variables you want to visualize
    ncols=3,
    size=5,
    wspace=0.5,                 # 🔥 smaller = lighter file
    frameon=True,
    legend_loc='right',
    show=False,
    return_fig=True           # 🔥 IMPORTANT
)

# =========================
# 🔥 RASTERIZE POINTS (KEY FIX)
# =========================
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

# =========================
# CLEAN AXES + BORDER
# =========================
for ax in fig.axes:
 
    ax.title.set_fontweight("normal")
    ax.xaxis.label.set_fontweight("normal")
    ax.yaxis.label.set_fontweight("normal")
    
    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_1_plots/SI/pandc_mesenchymal_leidenslusters_scvi_dbdr"

fig.savefig(f"{out_base}.pdf", dpi=600, bbox_inches="tight")  # 🔥 BEST for Illustrator
fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
# Cluster 7 flagged for removal based on low gene complexity and
# ambiguous marker gene profile
remove_clusters = ["7"]
mesenchymal_dr = mesenchymal_dr[~mesenchymal_dr.obs["mesenchymal_leiden"].isin(remove_clusters)].copy()

In [ ]:
mesenchymal_dr

In [ ]:
sc.tl.rank_genes_groups(mesenchymal_dr, "mesenchymal_leiden", use_raw=False)
sc.pl.rank_genes_groups(mesenchymal_dr, gene_symbols='features')

In [ ]:
# Clusters 0, 1, 4, 5, 6 share similar marker profiles (broad fibroblast/mesenchymal)
# Merged into a single supercluster: MSL1
# Convert to string if cluster labels are integers
mesenchymal_dr.obs["mesenchymal_leiden"] = mesenchymal_dr.obs["mesenchymal_leiden"].astype(str)

# Define the clusters to merge and the new label
to_merge = ["0", "1", "4", "5", "6"]
new_label = "MSL1"

# Apply the merge
mesenchymal_dr.obs["mesenchymal_leiden"] = mesenchymal_dr.obs["mesenchymal_leiden"].replace(to_merge, new_label)

In [ ]:
sc.tl.rank_genes_groups(mesenchymal_dr, "mesenchymal_leiden", use_raw=False)
sc.pl.rank_genes_groups(mesenchymal_dr, gene_symbols='features')

In [ ]:
# Clusters 2 and 3 share a distinct marker profile; merged into MSL2
mesenchymal_dr.obs["mesenchymal_leiden"] = mesenchymal_dr.obs["mesenchymal_leiden"].astype(str)

# Define the clusters to merge and the new label
to_merge = ["3", "2"]
new_label = "MSL2"

# Apply the merge
mesenchymal_dr.obs["mesenchymal_leiden"] = mesenchymal_dr.obs["mesenchymal_leiden"].replace(to_merge, new_label)

In [ ]:
# Visualize final MSL1/MSL2 labels with QC metrics on CONCORD UMAP

color_by = ['mesenchymal_leiden', 'doublet', 'cell_type', 'total_counts', 'n_genes_by_counts'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    mesenchymal_dr, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=3, font_size=6, point_size=5, legend_loc='upper right')

In [ ]:
mesenchymal_dr.obs['mesenchymal_leiden'].value_counts()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import scanpy as sc

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "font.weight": "normal", 
    "axes.titleweight": "normal",   # 🔥 no bold
    "axes.labelweight": "normal", 
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})
plt.rcParams["figure.figsize"] = (3, 3)
# =========================
# PLOT (let scanpy build grid)
# =========================
fig = sc.pl.embedding(
    mesenchymal_dr,
    basis="Concord_UMAP",
    color = ['mesenchymal_leiden', 'doublet_score', 'cell_type', 'total_counts', 'n_genes_by_counts', 'pct_counts_ribo'], # Choose which variables you want to visualize
    ncols=3,
    size=5,
    wspace=0.5,                 # 🔥 smaller = lighter file
    frameon=True,
    legend_loc='right',
    show=False,
    return_fig=True           # 🔥 IMPORTANT
)

# =========================
# 🔥 RASTERIZE POINTS (KEY FIX)
# =========================
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

# =========================
# CLEAN AXES + BORDER
# =========================
for ax in fig.axes:
 
    ax.title.set_fontweight("normal")
    ax.xaxis.label.set_fontweight("normal")
    ax.yaxis.label.set_fontweight("normal")
    
    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_1_plots/SI/pandc_mesenchymal_leidenslusters_finalumap_dr"

fig.savefig(f"{out_base}.pdf", dpi=600, bbox_inches="tight")  # 🔥 BEST for Illustrator
fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import scanpy as sc

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "font.weight": "normal", 
    "axes.titleweight": "normal",   # 🔥 no bold
    "axes.labelweight": "normal", 
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})
plt.rcParams["figure.figsize"] = (3, 3)
# =========================
# PLOT (let scanpy build grid)
# =========================
fig = sc.pl.embedding(
    mesenchymal_dr,
    basis="X_umap",
    color = ['mesenchymal_leiden', 'doublet_score', 'cell_type', 'total_counts', 'n_genes_by_counts', 'pct_counts_ribo'], # Choose which variables you want to visualize
    ncols=3,
    size=5,
    wspace=0.5,                 # 🔥 smaller = lighter file
    frameon=True,
    legend_loc='right',
    show=False,
    return_fig=True           # 🔥 IMPORTANT
)

# =========================
# 🔥 RASTERIZE POINTS (KEY FIX)
# =========================
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

# =========================
# CLEAN AXES + BORDER
# =========================
for ax in fig.axes:
 
    ax.title.set_fontweight("normal")
    ax.xaxis.label.set_fontweight("normal")
    ax.yaxis.label.set_fontweight("normal")
    
    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_1_plots/SI/pandc_mesenchymal_leidenslusters_scvifinallumap_dr"

fig.savefig(f"{out_base}.pdf", dpi=600, bbox_inches="tight")  # 🔥 BEST for Illustrator
fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
import scanpy as sc

# define colors for ALL categories that exist
cats = list(mesenchymal_dr.obs["cell_type"].astype("category").cat.categories)

# make everything grey, then override mesenchymal cell to yellow
celltype_colors = {c: "#808080" for c in cats}      # grey default
celltype_colors["mesenchymal cell"] = "#FFD700"     # yellow

mesenchymal_dr.uns["cell_type_colors"] = [celltype_colors[c] for c in cats]

# plot just cell_type with scanpy (uses the same embedding)
sc.pl.embedding(
    mesenchymal_dr,
    basis="Concord_UMAP",
    color=["cell_type"],
    size=10,
    legend_loc=None,
    frameon=False
)

In [ ]:
fig = plt.figure(figsize=(1.5, 1))

# We leave a tiny 1% margin so the black border line doesn't get clipped
ax = fig.add_axes([0.01, 0.01, 0.98, 0.98]) 

sc.pl.embedding(
    mesenchymal_dr,
    basis="Concord_UMAP",
    color="mesenchymal_leiden",
    size=4,
    frameon=True,  # 🔥 TURN THE FRAME BACK ON
    legend_loc='upper right',
    ax=ax,
    show=False,
    sort_order=True
)

# INSTEAD OF set_axis_off(), we just hide the text/notches:
ax.set_title("")
ax.set_xticks([])
ax.set_yticks([])
ax.set_ylabel("")
ax.set_xlabel("")


# Optional: Ensure the border line weight matches your Arial 8pt style
for spine in ax.spines.values():
    spine.set_linewidth(1)

# =========================
# 3. EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Fig_plots/UMAP_with_frame_concord_mesenchumalcelltype_dr"

# We still use pad_inches=0 to keep the file size tight to the box
plt.savefig(f"{out_base}.pdf", pad_inches=0, dpi=600)
plt.savefig(f"{out_base}.svg", format="svg", dpi=600, pad_inches=0)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "font.weight": "bold",
    "axes.titlesize": 9,
    "font.style": "normal",
    "font.variant": "normal",
    "font.stretch": "normal",  
    "axes.labelsize": 8,
    "axes.titleweight": "bold",
    "axes.labelweight": "bold",
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "axes.linewidth": 1,
    "lines.linewidth": 1,
    "xtick.major.width": 1,
    "ytick.major.width": 1,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})
# =========================
# FIGURE + TWO AXES
# =========================
fig, axes = plt.subplots(1, 2, figsize=(3, 1))  # 2 panels

# =========================
# PLOT EACH GENE SET SEPARATELY
# =========================
genesets = ["ECM_Matrisome", "Basement_Membrane"]

for ax, gs in zip(axes, genesets):

    sc.pl.embedding(
        mesenchymal_dr,
        basis="Concord_UMAP",
        color=gs,
        cmap='viridis',
        size=4,
        frameon=True,
        legend_loc='upper right',
        ax=ax,
        show=False,
        sort_order=True, 
        vmax=0.6
    )

    # clean axes (no ticks/labels)
    ax.set_title("")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

    # consistent border
    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =========================
# LAYOUT
# =========================
plt.tight_layout()

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Fig_plots/UMAP_with_frame_concord_mesenchumalbmscores_dr"

plt.savefig(f"{out_base}.pdf", dpi=600)
plt.savefig(f"{out_base}.svg", format="svg", dpi=600)



In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE (Illustrator-friendly)
# =========================
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["font.family"] = "Arial"

# =========================
# GENES
# =========================
color_by = ['LUM', 'ADIRF', 'RGS5', 'COL1A2', 'FABP4', 'S100A4']  # 🔥 fixed S100A4 typo

# =========================
# FORCE FIGURE
# =========================
plt.figure(figsize=(4, 3))  # adjust as needed

# =========================
# PLOT
# =========================
sc.pl.embedding(
    mesenchymal_dr,
    basis="Concord_UMAP",
    color=color_by,
    size=8,                 # 🔥 smaller = lighter file
    vmax=2,
    ncols=3,
    frameon=False,
    show=False,
    cmap='cool',
    use_raw=False
    # gene_symbols='features'  # ⚠️ only include if this column exists
)

# =========================
# GET FIGURE
# =========================
fig = plt.gcf()

# =========================
# 🔥 RASTERIZE POINTS (KEY)
# =========================
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

# =========================
# SAVE
# =========================
out_base = "/Users/kmeneses/Desktop/Fig_plots/Mesenchymal_UMAP_markers_dr"

fig.savefig(f"{out_base}.pdf", dpi=600, bbox_inches="tight")   # 🔥 BEST
fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")



In [ ]:
# EXPORT PRISM-FORMATTED EXPRESSION TABLES 
# Extract per-cluster expression of key collagen genes (COL4A1, COL4A2, COL1A1, COL1A2)
# Exported in wide format for GraphPad Prism violin/box plot input

genes = ["COL4A1", "COL4A2", "COL1A1", "COL1A2"]

import pandas as pd


# get expression matrix
expr = mesenchymal_dr[:, genes].to_df()

# add cluster labels
expr["mesenchymal_leiden"] = mesenchymal_dr.obs["mesenchymal_leiden"].values
prism_tables = {}

for gene in genes:
    
    df = expr[[gene, "mesenchymal_leiden"]].copy()
    
    prism = pd.DataFrame()
    
    for cluster in sorted(df["mesenchymal_leiden"].unique()):
        prism[f"{gene}_cluster_{cluster}"] = (
            df.loc[df["mesenchymal_leiden"] == cluster, gene]
            .reset_index(drop=True)
        )
    
    prism_tables[gene] = prism
    
    prism.to_csv(
        f"/Users/kmeneses/Desktop/Fig_plots/{gene}_mesenchyme_leiden_PRISM_dr.csv",
        index=False
    )

print("Prism tables exported.")

sc.pl.violin(
    mesenchymal_dr,
    genes,
    groupby="mesenchymal_leiden",
    stripplot=False,
    jitter=False,
    use_raw=False
)

In [ ]:
color_by = ['S100A6', 'LUM', 'DCN', 'RGS5', 'ADIRF', 'FABP4', 'ADAMTS4', 'PDGFRB' , 'C11orf96'] # Choose which variables you want to visualize

dp = sc.pl.dotplot(
    mesenchymal_dr,
    color_by,
    cmap="magma",
    groupby='mesenchymal_leiden',
    figsize=(4.5,1.5),
    use_raw=False, 
    gene_symbols='features',
    return_fig=True
)

dp.make_figure()

# now fig exists
dp.fig.subplots_adjust(wspace=0.8, left=0.25)
dp.savefig(
    "/Users/kmeneses/Desktop/Fig_plots/mesenchyme_marker_dotplot_dr.svg",
    dpi=600
)

dp.show()

In [ ]:
# Differential expression between MSL1 and MSL2 using Wilcoxon rank-sum test
# Results stored under key "mes_DEG_log1p" for downstream extraction


# adata_pancdb_mes contains only mesenchymal cells
sc.tl.rank_genes_groups(
    mesenchymal_dr,
    groupby="mesenchymal_leiden",   # e.g. 'Pericyte', 'Fibroblast'
    method="wilcoxon",
    use_raw=False,
    key_added="mes_DEG_log1p"
)

deg = mesenchymal_dr.uns["mes_DEG_log1p"]

# Extract top N genes per subtype
def get_deg_list(group, n=50):
    return pd.Series(deg["names"][group][:n]).tolist()

cluster0sig_n  = get_deg_list("MSL1")
cluster1sig_n     = get_deg_list("MSL2")

In [ ]:
import scanpy as sc
import pandas as pd

# Extract full DEG table (all clusters)
deg_df = sc.get.rank_genes_groups_df(
    mesenchymal_dr,
    key="mes_DEG_log1p",
    group=None   # <-- this gets ALL clusters
)

# Save full table
deg_df.to_csv(
    "/Users/kmeneses/Desktop/Fig_plots/mesenchymal_all_DEGs_dr.csv",
    index=False
)

In [ ]:
# Z-score normalized heatmap of top 50 MSL1 DEGs across MSL1 and MSL2
# Highlights genes specifically enriched in the MSL1 subpopulation

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

sc.pl.heatmap(
    mesenchymal_dr,
    var_names=cluster0sig_n,
    groupby="mesenchymal_leiden",
    cmap="viridis",
    standard_scale='var',   # raw log1p
    swap_axes=False,
    figsize=(8,2),
    use_raw=False,             # or 2.5 if your data extends higher
    show=False
)

fig = plt.gcf()

fig.savefig(
    "/Users/kmeneses/Desktop/Fig_plots/BM_gene_set_heatmap_0v1_dr.svg",
    bbox_inches="tight"
)

In [ ]:
# Z-score normalized heatmap of top 50 MSL1 DEGs across MSL1 and MSL2
# Highlights genes specifically enriched in the MSL1 subpopulation
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})



sc.pl.heatmap(
    mesenchymal_dr,
    var_names=cluster1sig_n,
    groupby="mesenchymal_leiden",
    cmap="viridis",
    standard_scale='var',   # <-- no z-score
    swap_axes=False,
    figsize=(8,2),
    use_raw=False,         # assuming .X is log1p
    show=False,    # <-- remove legend
)

fig = plt.gcf()

fig.savefig(
    "/Users/kmeneses/Desktop/Fig_plots/BM_gene_set_heatmap_1v0_dr.svg",
    bbox_inches="tight",
    dpi=600
)

In [ ]:
color = ['PLVAP', 'PECAM1']
sc.pl.dotplot(mesenchymal_dr, var_names=color, groupby='mesenchymal_leiden', use_raw=False)

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

sc.pl.umap(
    mesenchymal_dr,
    color="mesenchymal_leiden",
    show=True
)



In [ ]:
# MEAN ECM / BM SCORES PER MESENCHYMAL CLUSTER 
# Compute mean Basement_Membrane and ECM_Matrisome scores per cluster
# Quantifies which mesenchymal subtype drives ECM/BM program expression# SETTINGS
# -----------------------------
CLUSTER_COL = "mesenchymal_leiden"   # or "clusters_pdgfrb", "pdgfrb_clusters", etc.
BM_SCORE = "Basement_Membrane"
ECM_SCORE=["ECM_Matrisome"],
# clusters of interest

# -----------------------------
# SUBSET TO CLUSTERS

# -----------------------------
# MEAN BM SCORE PER CLUSTER
# -----------------------------
mean_bm = (
    mesenchymal_dr.obs
    .groupby(CLUSTER_COL)[BM_SCORE]
    .mean()
)

print("\nMean BM score per cluster:")
print(mean_bm)

In [ ]:
# -----------------------------
# SETTINGS
# -----------------------------
# SETTINGS
# -----------------------------
CLUSTER_COL = "mesenchymal_leiden"   # or "clusters_pdgfrb", "pdgfrb_clusters", etc.
BM_SCORE = "Basement_Membrane"
ECM_SCORE=["ECM_Matrisome"]
# clusters of interest

# -----------------------------
# SUBSET TO CLUSTERS

# -----------------------------
# MEAN BM SCORE PER CLUSTER
# -----------------------------
mean_ecm = (
    mesenchymal_dr.obs
    .groupby(CLUSTER_COL)[ECM_SCORE]
    .mean()
)

print("\nMean BM score per cluster:")
print(mean_ecm)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import scanpy as sc
from matplotlib.colors import ListedColormap, BoundaryNorm

# ------------------------------------------------------------
# WHY YOUR PLOT IS STILL NOT GREY:
# ccd.pl.plot_embedding() is ignoring adata.uns["<cat>_colors"].
# So the safest fix is: plot with scanpy directly (it WILL obey colors),
# using the same Concord_UMAP basis already stored in .obsm.
# ------------------------------------------------------------

# 1) Build plotting label: highlight two, grey everything else
highlight_cts = ["mesenchymal cell", "endothelial cell"]
key = "cell_type_plot"

s = hadata.obs["cell_type"].astype(str).copy()
s[~s.isin(highlight_cts)] = "Other"

ordered_cats = ["Other", "mesenchymal cell", "endothelial cell"]
cats_present = [c for c in ordered_cats if c in set(s.values)]
hadata.obs[key] = pd.Categorical(s, categories=cats_present, ordered=True)

# 2) Define exact colors (ORDER MUST MATCH categories)
color_map = {
    "Other": "#D3D3D3",            # grey
    "mesenchymal cell": "#F1C40F", # yellow
    "endothelial cell": "#1DDAE7", # cyan
}
palette = [color_map[c] for c in hadata.obs[key].cat.categories]

print("Category order:", list(hadata.obs[key].cat.categories))
print("Palette order :", palette)

# 3) Plot with scanpy (uses the same embedding coordinates)
#    IMPORTANT: basis must match the key in pancdb_adata.obsm
#    (you used 'Concord_UMAP' in ccd, so we use that here too).
mpl.rcParams.update({
    "font.family": "Arial",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

sc.pl.embedding(
    hadata,
    basis="X_umap",
    color=key,
    palette=palette,
    size=5,
    frameon=False,
    legend_loc="none",
    show=False
)

fig = plt.gcf()
ax = plt.gca()
ax.set_title("")  # clean title (optional)

# 4) Save high-res / Adobe-friendly
out_base = "/Users/kmeneses/Desktop/Fig_plots/UMAP_mes_escvinewndo_greyed"
fig.savefig(f"{out_base}.svg", bbox_inches="tight")
fig.savefig(f"{out_base}.pdf", bbox_inches="tight")
fig.savefig(f"{out_base}.png", dpi=900, bbox_inches="tight")

plt.show()

## 18. BM vs ECM Gene Set Z-Score Comparison Across Cell Types

To compare the relative enrichment of Basement Membrane (BM) and ECM_Matrisome 
programs across all pancreatic cell types, we computed direct mean expression scores 
from the raw expression matrix rather than using sc.tl.score_genes(). 

For each gene set, we:
1. Filtered to genes present in the dataset
2. Extracted the expression submatrix and computed the per-cell mean expression
3. Z-score normalized scores across all cells to enable cross-gene-set comparison

Results were visualized as a grouped boxplot with cell type on the x-axis and 
z-scored enrichment on the y-axis, with BM and ECM shown as separate hue groups. 
This allows direct visual comparison of which cell types are most enriched for 
BM versus broader ECM programs.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import zscore
import matplotlib as mpl

# =========================
# STYLE (optional but recommended)
# =========================
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["font.family"] = "Arial"

# --- Define your gene sets
gene_sets = {
    "BM": bmall_genes,
    "ECM": ecm_filtered
}

# --- 1) Filter valid genes
for gs_name, gs_genes in gene_sets.items():
    valid_genes = [g for g in gs_genes if g in hadata_clean_removed.var_names]
    gene_sets[gs_name] = valid_genes
    print(f"{gs_name}: {len(valid_genes)} genes used")

# --- 2) Compute mean expression scores
for gs_name, gs_genes in gene_sets.items():
    if len(gs_genes) == 0:
        continue
    
    idx = hadata_clean_removed.var_names.get_indexer(gs_genes)
    X_sub = hadata_clean_removed.X[:, idx]

    if hasattr(X_sub, "toarray"):
        X_sub = X_sub.toarray()

    hadata_clean_removed.obs[f"{gs_name}_score"] = X_sub.mean(axis=1)

# --- 3) Compute Z-scores
for gs_name in gene_sets:
    score_col = f"{gs_name}_score"
    if score_col in hadata_clean_removed.obs:
        hadata_clean_removed.obs[f"{gs_name}_zscore"] = zscore(
            hadata_clean_removed.obs[score_col], nan_policy='omit'
        )

# --- 4) Prepare dataframe
plot_cols = ["BM_zscore", "ECM_zscore", "cell_type"]
df_plot = hadata_clean_removed.obs[plot_cols].dropna().copy()

df_melted = df_plot.melt(
    id_vars="cell_type",
    var_name="GeneSet",
    value_name="Z-Score"
)

df_melted["GeneSet"] = df_melted["GeneSet"].str.replace("_zscore", "", regex=False)

# --- 5) Plot
fig, ax = plt.subplots(figsize=(10, 6))  # 🔥 use fig handle

sns.boxplot(
    data=df_melted,
    x="cell_type",
    y="Z-Score",
    hue="GeneSet",
    ax=ax
)

ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.set_title("BM vs ECM Z-Scores Across Cell Types")

ax.legend(title="Gene Sets", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()

# =========================
# 🔥 SAVE
# =========================
out_base = "/Users/kmeneses/Desktop/Fig_plots/BM_vs_ECM_boxplot"

fig.savefig(f"{out_base}.pdf", dpi=600, bbox_inches="tight")  # best for Illustrator
fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
hadata.var.index.name = None

In [ ]:
# remove problematic index name
hadata_clean.var.index.name = None

# OPTIONAL: drop redundant column if not needed
# hadata_clean.var = hadata_clean.var.drop(columns=["features"])

# save
hadata_clean.write("/Users/kmeneses/Desktop/Fig_plots/pancdb_clean_final.h5ad")

In [ ]:
# remove problematic index name
hadata_clean_removed.var.index.name = None

# OPTIONAL: drop redundant column if not needed
# hadata_clean.var = hadata_clean.var.drop(columns=["features"])

# save
hadata_clean_removed.write("/Users/kmeneses/Desktop/Fig_plots/pancdb_clean_final_really.h5ad")

In [ ]:
# remove problematic index name
mesenchymal.var.index.name = None

# OPTIONAL: drop redundant column if not needed
# hadata_clean.var = hadata_clean.var.drop(columns=["features"])

# save
mesenchymal_dr.write("/Users/kmeneses/Desktop/Fig_plots/pancdb_mesenchymal_clean_final_dr.h5ad")

In [ ]:
# remove problematic index name
mesenchymal_endopancdb.var.index.name = None

# OPTIONAL: drop redundant column if not needed
# hadata_clean.var = hadata_clean.var.drop(columns=["features"])

# save
mesenchymal_endopancdb.write("/Users/kmeneses/Desktop/Fig_plots/pancdb_mesenchymalECs_clean_final.h5ad")

In [ ]:
ECM = ['SFN', 'COL4A3', 'COL18A1', 'COL15A1', 'COL5A3', 'COL6A3', 'COL6A2', 'COL5A2', 'COL1A1', 'COL1A2', 'COL6A1', 'COL3A1', 
       'LGALS4', 'LOXL4', 'SFRP2', 'SPARCL1', 'IGFBP7', 'AGPS', 'AGRN', 'CLASRP', 'LUM', 'MGP', 'AMBP', 'HSPG2', 
       'F13A1', 'ASPN', 'LAMC3', 'TINAGL1', 'VWA1', 'PCOLCE', 'MFGE8', 'FBLN2', 
       'LAMB2', 'FBN1', 'LAMA2', 'THBS1', 'VTN', 'FN1', 'FGG', 'FGB', 'FGA', 'LAMA5', 'COL16A1', 'COL11A1', 
       'COL4A2', 'COL4A1', 'COL2A1', 'OGN', 'EMILIN1', 'LAMA4', 'POSTN', 'FBN2', 'COL12A1', 'COL14A1', 'COL5A1', 'LAMC1', 'LAMC2',  'LAMB3', 'FBLN1',
        'LAMB1', 'LAMB4', 'LAMA3', 'SDC1']

## 19. Cross-Dataset Mesenchymal Signature Transfer and Integration

To validate the MSL1 (fibroblast-like) and MSL2 (pericyte-like) mesenchymal 
subpopulations identified in PancDB, we projected their transcriptional signatures 
onto three independent public pancreatic datasets:

- **Human Cell Atlas (HCA)** — pancreatic stellate cells
- **Tabula Sapiens (TS)** — pancreatic fibroblasts and stellate cells  
- **Shapiro EC Atlas (ShpECAtlas)** — pericytes and fibroblasts from a vascular atlas

For each dataset, mesenchymal/stromal cells were isolated, normalized, and scored 
for the MSL1 (Fibro_signature) and MSL2 (Pericyte_signature) gene lists derived 
from the PancDB Wilcoxon DEG analysis. Scores were visualized on each dataset's 
native UMAP embedding to assess cross-dataset concordance.

All four mesenchymal datasets were then merged into a single AnnData object, 
gene names cleaned of ENSG suffixes and duplicate entries, and re-integrated 
using CONCORD with donor_id as the batch key. The resulting joint embedding was 
used to visualize shared cell type structure across datasets.

A curated panel of canonical mesenchymal marker genes (ven_diagram gene list) 
was visualized as a matrix plot grouped by dataset and cell type, with PancDB 
cells split by MSL1/MSL2 Leiden cluster labels. ECM_Matrisome and 
Basement_Membrane gene set scores were also computed on the merged object 
and visualized as a dot plot.

In [ ]:
# Human Cell Atlas pancreatic stellate cell dataset
# Used to validate PancDB mesenchymal signatures in an independent cohort
hca_mesenchyme=sc.read_h5ad('/Volumes/Samsung_4TB/Public_human_RNAdasets/humancellatlas_pancreas_stellatecells.h5ad')
hca_mesenchyme

In [ ]:
color_by = ['cell_type', 'Dataset'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    hca_mesenchyme, basis='X_umap', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=15, legend_loc='upper right')

In [ ]:
import scanpy as sc

# 1) Preserve raw counts if not already stored
hca_mesenchyme.raw = hca_mesenchyme.copy()   # keeps raw counts in .raw.X

# 2) Normalize total counts per cell
sc.pp.normalize_total(hca_mesenchyme, target_sum=1e4)

# 3) Log1p transform
sc.pp.log1p(hca_mesenchyme)

# 4) Store the log1p matrix in a layer
hca_mesenchyme.layers["log1p"] = hca_mesenchyme.X.copy()

In [ ]:
print(hca_mesenchyme.layers["log1p"])

In [ ]:
# Example: set var_names to gene symbols stored in adata.var["feature_name"]
hca_mesenchyme.var_names = hca_mesenchyme.var["feature_name"].astype(str).values

# Ensure uniqueness (important!)
hca_mesenchyme.var_names_make_unique()

In [ ]:
# adata_other = your MERSCOPE or other dataset

sc.tl.score_genes(hca_mesenchyme, cluster1sig_n, layer='log1p', score_name="Pericyte_signature")
sc.tl.score_genes(hca_mesenchyme, cluster0sig_n, layer='log1p', score_name="Fibro_signature")

In [ ]:

# Plot the UMAP embeddings
color_by = ['Pericyte_signature', 'Fibro_signature'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    hca_mesenchyme, basis='X_umap', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='on data', 
    save_path='/Users/kmeneses/Desktop/SI_plots/hcamesenchyme_pancdbcellsigtransferumap.png')

In [ ]:
# Tabula Sapiens pancreas dataset — contains fibroblasts and stellate cells
ts_adata=sc.read_h5ad('/Volumes/Samsung_4TB/Public_human_RNAdasets/TS_Pancreas.h5ad')
ts_adata

In [ ]:
# Inspect donor and cell type distribution on scVI UMAP

color_by = ['donor', 'cell_ontology_class'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    ts_adata, basis='X_scvi_umap', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=15, legend_loc='upper right')

In [ ]:
color_by = ['free_annotation', 'cell_ontology_class', 'manually_annotated'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    ts_adata, basis='X_scvi_umap', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=15, legend_loc='upper right', 
    save_path='/Users/kmeneses/Desktop/SI_plots/ts_celltypedonoris_scviumap.png')

In [ ]:
# Column cell_ontology_class clusters
CLUSTER_KEY = "cell_ontology_class"    # <-- change if needed

# Clusters you want to merge into a separate AnnData
keep_clusters = ["fibroblast", 'pancreatic stellate cell']     # <-- EDIT: choose the two clusters you want

# Subset + merge them into a new AnnData
ts_mesenchyme = ts_adata[ts_adata.obs[CLUSTER_KEY].astype(str).isin(keep_clusters)].copy()

print("Cells in merged subset:", ts_mesenchyme.n_obs)

In [ ]:
ts_mesenchyme.obs = ts_mesenchyme.obs.rename(columns={"cell_ontology_class": "cell_type"})

In [ ]:
ts_mesenchyme.obs = ts_mesenchyme.obs.rename(columns={"donor": "donor_id"})

In [ ]:
color_by = ['donor_id', 'cell_type'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    ts_mesenchyme, basis='X_umap', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=15, legend_loc='upper right', 
    save_path='/Users/kmeneses/Desktop/SI_plots/ts_mesenymleonly_donorcelltype.png')

In [ ]:
import scanpy as sc

# 1) Preserve raw counts if not already stored
ts_mesenchyme.raw = ts_mesenchyme.copy()   # keeps raw counts in .raw.X

# 2) Normalize total counts per cell
sc.pp.normalize_total(ts_mesenchyme, target_sum=1e4)

# 3) Log1p transform
sc.pp.log1p(ts_mesenchyme)

# 4) Store the log1p matrix in a layer
ts_mesenchyme.layers["log1p"] = ts_mesenchyme.X.copy()

In [ ]:
# adata_other = your MERSCOPE or other dataset

sc.tl.score_genes(ts_mesenchyme, cluster1sig_n, layer='log1p', score_name="Pericyte_signature")
sc.tl.score_genes(ts_mesenchyme, cluster0sig_n, layer='log1p', score_name="Fibro_signature")

In [ ]:
color_by = ['Pericyte_signature', 'Fibro_signature'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    ts_mesenchyme, basis='X_umap', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=15, legend_loc='upper right', 
    save_path='/Users/kmeneses/Desktop/SI_plots/tsmesenchyme_cellsigpancdbtransfer.png')

In [ ]:
# Shapiro endothelial cell atlas — contains pericytes and fibroblasts
# Subset to mesenchymal populations for cross-dataset comparison
shp_adata=sc.read_h5ad('/Volumes/Samsung_4TB/Desktop_copy/07092025/shp_mesencymevasc.h5ad')
shp_adata

In [ ]:
# Column containing clusters
CLUSTER_KEY = "cell_type"    # <-- change if needed

# Clusters you want to merge into a separate AnnData
keep_clusters = ["Pericyte", 'Fibroblasts']     # <-- EDIT: choose the two clusters you want

# Subset + merge them into a new AnnData
shpmesenchyme_adata = shp_adata[shp_adata.obs[CLUSTER_KEY].astype(str).isin(keep_clusters)].copy()

print("Cells in merged subset:", shpmesenchyme_adata.n_obs)

In [ ]:
# Column containing clusters
CLUSTER_KEY = "cell_type"

# Clusters to merge
keep_clusters = ["Pericyte", "Fibroblasts"]

# Subset those cells
shpmesenchyme_adata = shp_adata[
    shp_adata.obs[CLUSTER_KEY].astype(str).isin(keep_clusters)
].copy()

# Add merged label
shpmesenchyme_adata.obs["cell_type_merged"] = "Stromal Cells"

# (Optional) keep original labels too
shpmesenchyme_adata.obs["cell_type_original"] = shpmesenchyme_adata.obs[CLUSTER_KEY].astype(str)

print("Cells in merged stromal subset:", shpmesenchyme_adata.n_obs)
print(shpmesenchyme_adata.obs["cell_type_merged"].value_counts())

In [ ]:
color_by = ['donor_id', 'cell_type'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    shpmesenchyme_adata, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=15, legend_loc='upper right', 
    save_path='/Users/kmeneses/Desktop/SI_plots/shp_craigperifibro_concordumap.png')

In [ ]:
# adata_other = your MERSCOPE or other dataset

sc.tl.score_genes(shpmesenchyme_adata, cluster1sig_n, score_name="Pericyte_signature")
sc.tl.score_genes(shpmesenchyme_adata, cluster0sig_n, score_name="Fibro_signature")

In [ ]:
color_by = ['Pericyte_signature', 'Fibro_signature'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    shpmesenchyme_adata, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=15, legend_loc='upper right', vmax=1, 
    save_path='/Users/kmeneses/Desktop/SI_plots/shp_craigmesenchyme_celltransferscores_cumap.png')

In [ ]:
ts_mesenchyme.var_names

In [ ]:
# Concatenate TS, HCA, ShpECAtlas, and PancDB mesenchymal objects
# Add dataset labels and make obs_names unique to avoid index collisions

# Your three AnnData objects (already mesenchymal-only)
ad1 = ts_mesenchyme   # PanC-DB
ad2 = hca_mesenchyme   # Tabula
ad3 = shpmesenchyme_adata 
ad4 = mesenchymal  # MERSCOPE

# Add dataset labels + make obs_names unique
def prep(a, name):
    a = a.copy()
    a.obs_names = [f"{name}_{i}" for i in a.obs_names]
    a.obs["DBS"] = name
    return a

ad1 = prep(ad1, "TabulaS")
ad2 = prep(ad2, "HCA")
ad3 = prep(ad3, "ShpECAtlas")
ad4 = prep(ad4, "PancDB")


# Merge — keep only shared genes
merged_mesenchymepulbic = ad.concat([ad1, ad2, ad3, ad4], join="outer")

# Make dataset column categorical
merged_mesenchymepulbic.obs["DBS"] = merged_mesenchymepulbic.obs["DBS"].astype("category")

print("Merged AnnData:", merged_mesenchymepulbic.shape)
print(merged_mesenchymepulbic.obs["DBS"].value_counts())


In [ ]:
color_by = ['DBS', 'cell_type'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    merged_mesenchymepulbic, basis='X_umap', color_by=color_by, figsize=(5, 2.5), dpi=600, ncols=2, font_size=5, point_size=10, legend_loc='upper left', 
    save_path='/Users/kmeneses/Desktop/SI_plots/sdvsaconcordumap.svg')

In [ ]:
sc.pl.umap(
    merged_mesenchymepulbic,
    color="DBS",
    ncols=2,                 # adjust depending on layout
    frameon=False,
    size=6,
    legend_loc='upper left',         # remove repeated legends
    wspace=0.3
)

In [ ]:
merged_mesenchymepulbic.var_names

In [ ]:
import pandas as pd

# convert to list
var_list = list(merged_mesenchymepulbic.var_names)

# remove ENSG suffix
clean_genes = [g.split("_ENSG")[0] for g in var_list]

# remove duplicates while preserving order
clean_genes = list(pd.unique(clean_genes))

print("Original genes:", len(var_list))
print("Clean genes:", len(clean_genes))

In [ ]:
# convert var_names to cleaned symbols
merged_mesenchymepulbic.var_names = [
    g.split("_ENSG")[0] for g in merged_mesenchymepulbic.var_names
]

# ensure uniqueness (AnnData requirement)
merged_mesenchymepulbic.var_names_make_unique()

print("Total genes:", len(merged_mesenchymepulbic.var_names))

In [ ]:
[g for g in merged_mesenchymepulbic.var_names if "A2MP1" in g]

In [ ]:
import pandas as pd

# remove ENSG suffix
clean_names = [g.split("_ENSG")[0] for g in merged_mesenchymepulbic.var_names]

# find duplicates
keep = ~pd.Series(clean_names).duplicated()

# subset AnnData
merged_mesenchymepulbic = merged_mesenchymepulbic[:, keep].copy()

# set cleaned gene names
merged_mesenchymepulbic.var_names = pd.Index(clean_names)[keep]

print("Genes after cleaning:", len(merged_mesenchymepulbic.var_names))

In [ ]:
import pandas as pd

adata = merged_mesenchymepulbic

# get base gene name (remove -1, -2 etc.)
base_names = [g.split("-")[0] for g in adata.var_names]

# keep first occurrence
keep = ~pd.Series(base_names).duplicated()

# subset AnnData
adata = adata[:, keep].copy()

# restore clean names
adata.var_names = pd.Index(base_names)[keep]

print("Remaining genes:", len(adata.var_names))

In [ ]:
[g for g in adata.var_names if "GCG" in g]

In [ ]:
color_by = ['cell_type', 'DBS'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    merged_mesenchymepulbic, basis='X_umap', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=15, legend_loc='upper right', vmax=1, 
    save_path='/Users/kmeneses/Desktop/SI_plots/mergedmesenchyme_xumap.png')

In [ ]:
sc.pp.normalize_per_cell(merged_mesenchymepulbic)
sc.pp.log1p(merged_mesenchymepulbic)

# Select top 2000 highly variable features (Seurat v3 method)
# Run CONCORD with donor_id as batch key to correct for inter-dataset variation
# Produces a joint low-dimensional embedding across all four datasets

In [ ]:
# Set device to cpu or to gpu (if your torch has been set up correctly to use GPU), for mac you can use either torch.device('mps') or torch.device('cpu')
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# (Optional) Select top variably expressed/accessible features for analysis (other methods besides seurat_v3 available)
feature_list = ccd.ul.select_features(merged_mesenchymepulbic, n_top_features=2000, flavor='seurat_v3')


# If integrating data across batch, simply add the domain_key argument to indicate the batch key in adata.obs
cur_ccd = ccd.Concord(adata=merged_mesenchymepulbic, input_feature=feature_list, use_faiss=False, domain_key='donor_id', device=device) 

# Encode data, saving the latent embedding in adata.obsm['Concord']
cur_ccd.fit_transform(output_key='Concord_mergedmesenchyme')

In [ ]:
ccd.ul.run_umap(merged_mesenchymepulbic, source_key='Concord_mergedmesenchyme', result_key='Concord_mergedmesenchyme_UMAP', n_components=2, n_neighbors=15, min_dist=0.1, metric='euclidean')

# Plot the UMAP embeddings
color_by = ['cell_type', 'DBS'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    merged_mesenchymepulbic, basis='Concord_mergedmesenchyme_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=10, point_size=10, legend_loc='upper right', 
    save_path='/Users/kmeneses/Desktop/SI_plots/mergedmesenchymal_concord_umap.png')

In [ ]:
color_by = ['cell_type', 'DBS'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    merged_mesenchymepulbic, basis='Concord_mergedmesenchyme_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='upper right', 
    save_path='/Users/kmeneses/Desktop/SI_plots/mergedmesenchymal_concordumap_celltypedbs.png')

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

sc.pl.violin(
    merged_mesenchymepulbic,
    ["Pericyte_signature", "Fibro_signature"],
    groupby="DBS",
    stripplot=False,
    jitter=False,
    multi_panel=True,
    show=False
)

fig = plt.gcf()
fig.set_size_inches(14,4)   # make larger
for ax in fig.axes:
    ax.grid(False)

plt.savefig("/Users/kmeneses/Desktop/SI_plots/mesenchyme_merged_signatures_violin.pdf", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
sc.pl.heatmap(merged_mesenchymepulbic, ven_diagram, groupby=['DBS', 'cell_type'], swap_axes=True, show_gene_labels=True)

In [ ]:
# Hand-curated list of canonical mesenchymal identity genes spanning:
# BM collagens (COL4A1/2), fibrillar collagens (COL1A1/2, COL5A1/2, COL6A1/3),
# fibroblast markers (LUM, DCN, FN1), pericyte markers (RGS5, PDGFRB, CSPG4,
# NOTCH3, MCAM), smooth muscle (ACTA2, MYH11, TAGLN),
# and adipocyte-like markers (FABP4, ADIRF)
ven_diagram = ['LAMA2', 'LAMB2', 'COL4A1', 'COL4A2', 'COL6A3', 'COL5A2', 
'COL5A1', 'LUM', 'COL6A1', 'DCN', 'COL1A1', 'COL1A2', 'FN1', 'MCAM', 'NOTCH3', 'RGS5', 'CSPG4', 
'MMP2', 'SPARCL1', 'TPM2', 'PDGFRB', 'FABP4', 'ADIRF', 'ADAMTS9', 'ACTA2', 'MYH11', 'TAGLN', 'PDGFRA']


In [ ]:
merged_mesenchymepulbic

In [ ]:
unique_genes = list(pd.unique(ven_diagram))

In [ ]:
# For PancDB cells, replace cell type with MSL1/MSL2 Leiden cluster labels
# so PancDB subpopulations are shown at the same resolution as other datasets
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, leaves_list
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["font.family"] = "Arial"

# =========================
# COPY DATA
# =========================
adata = merged_mesenchymepulbic.copy()

DBS_COL = "DBS"
CELLTYPE_COL = "cell_type"
LEIDEN_COL = "mesenchymal_leiden"
GROUP_COL = "DBS_cell_type"

# =========================
# CREATE GROUP COLUMN
# =========================
adata.obs[GROUP_COL] = (
    adata.obs[DBS_COL].astype(str) + "_" +
    adata.obs[CELLTYPE_COL].astype(str)
)

# Replace PancDB with Leiden clusters
panc_mask = adata.obs[DBS_COL] == "PancDB"
adata.obs.loc[panc_mask, GROUP_COL] = (
    "PancDB_" +
    adata.obs.loc[panc_mask, LEIDEN_COL].astype(str)
)

# =========================
# FORCE GROUP ORDER
# =========================
cats = adata.obs[GROUP_COL].astype(str).unique().tolist()

panc_first = sorted([c for c in cats if c.startswith("PancDB")])
others = sorted([c for c in cats if not c.startswith("PancDB")])

order = panc_first + others

adata.obs[GROUP_COL] = pd.Categorical(
    adata.obs[GROUP_COL],
    categories=order,
    ordered=True
)

# =========================
# PREPARE GENES (FINAL LIST)
# =========================
genes = [g for g in ven_diagram if g in adata.var_names]

# =========================
# EXTRACT EXPRESSION (log1p layer)
# =========================
X = adata.layers["log1p"][:, [adata.var_names.get_loc(g) for g in genes]]
X = X.toarray() if hasattr(X, "toarray") else X

expr = pd.DataFrame(X, columns=genes)
expr[GROUP_COL] = adata.obs[GROUP_COL].values

# =========================
# MEAN EXPRESSION PER GROUP
# =========================
group_means = expr.groupby(GROUP_COL).mean().T  # genes x groups

# =========================
# CLUSTER GENES
# =========================
Z = linkage(group_means, method="average", metric="euclidean")
gene_order = group_means.index[leaves_list(Z)].tolist()

# =========================
# PLOT (FORCE FIGURE)
# =========================
plt.figure(figsize=(12, 4))

sc.pl.matrixplot(
    adata,
    var_names=gene_order,
    groupby=GROUP_COL,
    cmap="viridis",
    standard_scale=None,
    swap_axes=False,
    show=False
)

# =========================
# SAVE (ROBUST)
# =========================
fig = plt.gcf()

out_base = "/Users/kmeneses/Desktop/Fig_plots/stromal_matrixplot_highres"

fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")
fig.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")

plt.close(fig)

In [ ]:
# Build a composite grouping label: DBS_celltype
# For PancDB cells, replace cell type with MSL1/MSL2 Leiden cluster labels
# so PancDB subpopulations are shown at the same resolution as other datasets
# =========================
# FORCE NEW FIGURE
# =========================
plt.figure(figsize=(14, 4))

# =========================
# PLOT
# =========================
sc.pl.matrixplot(
    adata,
    var_names=gene_order,
    groupby=GROUP_COL,
    cmap="viridis",
    swap_axes=False,
    standard_scale=None,
    show=False   # 🔥 IMPORTANT
)

# =========================
# GET CURRENT FIGURE
# =========================
fig = plt.gcf()

# =========================
# SAVE
# =========================
out_base = "/Users/kmeneses/Desktop/Fig_plots/stromal_matrixplot_highres"

fig.savefig(f"{out_base}.pdf", dpi=600, bbox_inches="tight")
fig.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")  # backup format

plt.close(fig)  # 🔥 ensures file writes properly

In [ ]:
# Re-score ECM_Matrisome and Basement_Membrane gene sets on the merged object
# Dotplot grouped by both DBS and cell_type to compare ECM enrichment
# across datasets and cell type labels simultaneously

mpl.rcParams["svg.fonttype"] = "none"   # keeps text editable
mpl.rcParams["font.family"] = "Arial"   # optional but recommended
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

# Define properly as dictionary
gene_sets = {
    "ECM_Matrisome": ecm_filtered,
    "Basement_Membrane": bmall_genes
}

# Score gene sets
for gs_name, gene_list in gene_sets.items():

    valid_genes = [g for g in gene_list if g in adata.var_names]

    if len(valid_genes) == 0:
        print(f"⚠️ Skipping {gs_name} (no valid genes found)")
        continue

    sc.tl.score_genes(
        adata,
        gene_list=valid_genes,
        score_name=gs_name,
        use_raw=False
    )

# Collect successfully scored sets
scored_gene_sets = [
    gs for gs in gene_sets.keys()
    if gs in adata.obs.columns
]

print("Scored gene sets:", scored_gene_sets)

# Plot
dp = sc.pl.dotplot(
    adata,
    var_names=scored_gene_sets,
    groupby=["DBS", "cell_type"],
    title="Relative Enrichment of ECM Programs",
    colorbar_title="Z-score",
    cmap="viridis",
    standard_scale="var",
    return_fig=True   # <-- important
)

# Save as SVG
dp.savefig("/Users/kmeneses/Desktop/Fig_plots/ECM_gene_mergedmesenchyme_dotplot_clean.svg", dpi= 600, bbox_inches="tight")

